In [ ]:
print("[cell] 01 run parameters (kapi --env): environment variables of this r", flush=True)
# run parameters (kapi --env): environment variables of this run
import os
os.environ['LAB_FOLDS'] = ''
os.environ['LID_STRICT'] = '1'
os.environ['T3S'] = '1'
os.environ['CT_R2_ONLY'] = 'CA'
os.environ['T3S_PER_GPU'] = '1'
os.environ['NOREL'] = '1'

# FLAG 2027 - NB-174: RFSTUNE-01 report, nested choice of the RFS projector recipe on SAFE-04 (CPU)

**What (Experiment/RFSTUNE-01/README.md, registered 11/11 before any number).** U_<arm> = F1s + CA-FUSE(RFS streams of the arm) for the
15 registered arms of FLAG_173; nested arm choice SEL vs the control a0; rule: SEL - a0 on g cells >= +0.3, >= 4/5 folds, held-out
Urdu / Hindi >= -0.2. Validation only (SAFE-04, purged); no dev, no CB, no Evaluation, LID_STRICT rows, NOREL. Internet OFF.

In [ ]:
print("[boot] FLAG_184_factor_report | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
print("[cell] env", flush=True)

import os, sys, json, time, shutil, subprocess, hashlib, base64, io, zipfile, threading
from pathlib import Path
T_START = time.time()
LOCAL = os.environ.get("FLAG_LOCAL") == "1"
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORK = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
EXP = os.environ.get("LAB_EXP", "CLEANTUNE-01")
TAG = "ct01r2"
EXP_CODE = ["Experiment/CLEANTUNE-01/run.py", "Experiment/VOICELORA-01/run.py", "Experiment/FACELORA-05/run.py", "Experiment/FACELORA-03/run.py", "Experiment/AUXP-01/run.py", "Experiment/SCALE-CC-02/run.py", "Experiment/SCALE-CC-01/run.py", "Experiment/SCALE-CC-01/vox2b_groups.json", "Experiment/SCALE-CC-01/vox2_gender.json", "Experiment/DSEL-03/run.py", "Experiment/DSEL-02/run.py", "Experiment/T3RECIPE-01/run.py", "Experiment/T3RECIPE-01/t3r.py", "Experiment/DSEL-01/vox2_groups.json", "Experiment/EN-T3-01/run.py", "Experiment/CLEAN-EN-01/submission_clean_en01_dev.zip", "Experiment/CLEANTUNE-01/ct_round2.json"]                                            # the experiment's own code: a resume needs the same files
FINAL_AFTER_REPORT = True                                      # True: the final head depends on the report (a chosen variant)
FOLDS = [int(x) for x in os.environ.get("LAB_FOLDS", "1,2,3,4,5").split(",") if x]
SMOKE = os.environ.get("FL3Z_SMOKE") == "1"                       # wiring test: fold 1 only
FOLDS = FOLDS[:1] if SMOKE else FOLDS
os.environ["CT_GROUP"] = "r2"                                   # CLEANTUNE-01 group of this notebook
os.environ["BN42"] = "1"                                         # Bengali-clean rows + FLAG_109 adapters only
SPLIT = os.environ.get("E2E_SPLIT", "dev")                         # dev: flag2027-e2e-dev; eval: the components of a FLAG_50 run
BUDGET_H = float(os.environ.get("E2E_BUDGET_H", "10.5"))            # no new job starts after this many hours (Kaggle stops at 12)


def log(*a):
    print(time.strftime("%H:%M:%S"), f"[{(time.time() - T_START) / 3600:4.1f}h]", *a, flush=True)


mans = [(m.parent, json.loads(m.read_text())) for m in sorted(IN.rglob("lab_manifest.json"))]
main = [(d, m) for d, m in mans if m.get("kind", "data") == "data"]
assert len(main) == 1, f"attach exactly one flag2027-lab-data (found {len(main)})"
DATA, MAN = main[0]
ROOTS = [d for d, _ in mans]
eda = sorted(IN.rglob("eda_bundle_manifest.json"))
assert len(eda) == 1, f"attach exactly one flag2027-eda-bundle (found {len(eda)})"
created = json.loads(eda[0].read_text())["created"]
assert created == MAN["eda_bundle_created"], f"eda bundle {created} != the lab's {MAN['eda_bundle_created']}"
e2e = [(m.parent, json.loads(m.read_text())) for m in sorted(IN.rglob("e2e_manifest.json"))]
e2e = [(d, m) for d, m in e2e if m.get("split") == SPLIT and (d / f"components_{SPLIT}.npz").exists()]
assert len(e2e) >= 1, f"attach flag2027-e2e-dev (or, for E2E_SPLIT=eval, the output of FLAG_50 on the Evaluation data)"
E2E_DIR, E2E_MAN = e2e[0]
SP = WORK / "sp"; (SP / "out/safe04").mkdir(parents=True, exist_ok=True)
EXTRA_LINKS = {}                                                    # lab path -> folder made by this notebook (extraction cells)
OUTD = WORK / "e2e_out"; OUTD.mkdir(parents=True, exist_ok=True)
LAB = WORK / "lab"
import torch
NGPU = torch.cuda.device_count()
GPU = [torch.cuda.get_device_name(i) for i in range(NGPU)] or ["none (CPU: slow)"]
log(f"experiment {EXP} ({TAG}) | folds {FOLDS} | split {SPLIT} ({E2E_DIR}, built {E2E_MAN['created']}) | lab data built {MAN['created']} | "
    f"eda {created} | GPUs {GPU} | CPUs {os.cpu_count()} | torch {torch.__version__} | budget {BUDGET_H} h")

os.environ["T3S"] = "1"                                            # T3S-01 (Experiment/T3S-01/README.md)
os.environ.setdefault("CT_R2_ONLY", "CA")                          # the in-run F1s control: CA only (no 'b' copy under T3S)
assert os.environ.get("LID_STRICT") == "1", "T3S-01 runs on the LID_STRICT rows (README §1)"
T3S_PARTS = [p for p in os.environ.get("T3S_PARTS", "farl,clip,b3s,c5s").split(",") if p]   # one job per face
T3S_PER_GPU = int(os.environ.get("T3S_PER_GPU", "2"))             # projector steps are launch-bound: two jobs share a GPU
T3S_STAGGER = int(os.environ.get("T3S_STAGGER", "300"))           # seconds between job starts (RAM peak of the data loading)

## 1. Code (embedded) + data links + SAFE-04 reference tables

In [ ]:
print("[cell] setup: code + data links", flush=True)

# embedded code (= the local files, md5 below) -> /kaggle/working/lab; data folders linked next to it; SAFE-04 reference tables
CODE_B64 = "..."
CODE_MD5 = {
 "kaggle/build_l0_notebooks.py": "e2f63ea92aa5c62e66a034a20a5ab493",
 "Experiment/SAFE-04/lab_streams.py": "e288a49c7f77f9a22e9cce00aead459b",
 "Experiment/SAFE-04/rel_sweep.py": "7166b20f70f25f342a22bfcadc5883ac",
 "Experiment/SAFE-04/rel_sweep2.py": "75899e4caf38b38bbf953da6b5b62a04",
 "Experiment/SAFE-04/rel_sweep5.py": "557afcb255ab4f1b4696296633d87a88",
 "Experiment/SAFE-04/rel.py": "402aaf561639e75409f2ead79d78a939",
 "Experiment/VOX2-01/run_bridge.py": "e3260d3276cfdca8eb08dbcfd26f19aa",
 "Experiment/T3ENC-01/run.py": "fd780c6f94762c24b9d9cf4db49df181",
 "Experiment/ALLNEG-01/run.py": "413e0779662fc2be0b5fb75930129e5b",
 "Experiment/BIGB-01/run.py": "f2a9a4f099496905ea93d9cf44025860",
 "Experiment/SCALE-VB/run.py": "2e2913fb5310c53191ead982de51582e",
 "Experiment/VBSEL-01/run.py": "2e99d3b90dc56675ca4b36ded1a83e96",
 "Experiment/VBSEL-01/vox2_sa_speakers.json": "37f4e8670949bc32f086003a593f9036",
 "Experiment/ENBASE-01/run.py": "4a40fb17c9585c2f66b177f5b529b777",
 "Experiment/SAFE-04/stream_gate.py": "db5d4fbbd5917d4813bd96797aa61058",
 "Experiment/SAFE-04/lab_fuse.py": "fa8ce8f652bf3ee13597e872fda40b83",
 "Experiment/STACK-02/run.py": "3709cf4c175e60945144ec7c4198dbd0",
 "Experiment/STACK-02/pools.py": "cd867144bf43f0283003a8f1ba685cc0",
 "Experiment/STACK-02/eligible.json": "bbab2a531702ab631652d21e25a825d2",
 "Experiment/STACK-02/fold_persons.json": "aede157945600732e0f8a8337b8df0b5",
 "Experiment/E2E/e2e_lib.py": "f8efa92934ba217d2b1e6c665815a729",
 "Experiment/CLEANTUNE-01/run.py": "634b708ef21f63d4366f0bafc6daaaaa",
 "Experiment/VOICELORA-01/run.py": "ebc0836826d71cf539ece938cbbb7ea5",
 "Experiment/FACELORA-05/run.py": "9be8d927c170cbb439e466041259b762",
 "Experiment/FACELORA-03/run.py": "c4d09a9fa645ab5094cc78654bc535ac",
 "Experiment/AUXP-01/run.py": "900fe3e223b7edff837c579c817fd13a",
 "Experiment/SCALE-CC-02/run.py": "73869739d0eaa9b842741269545f122e",
 "Experiment/SCALE-CC-01/run.py": "e7be3e2bfa47c0e0af30e51daaa6400d",
 "Experiment/SCALE-CC-01/vox2b_groups.json": "57092daac915915414588062dff6caff",
 "Experiment/SCALE-CC-01/vox2_gender.json": "950a8ba9c44e60db1e9bb5b8aac8d4b7",
 "Experiment/DSEL-03/run.py": "54da2a723624ff1b9ce320e47edc8074",
 "Experiment/DSEL-02/run.py": "97acc6c949ecc0f146d6009a5f1929ee",
 "Experiment/T3RECIPE-01/run.py": "13cfbd6f8a5cc445b025de740c4bf1fb",
 "Experiment/T3RECIPE-01/t3r.py": "ba1034827f4d2c62a0afd3a3f6b8fe1f",
 "Experiment/DSEL-01/vox2_groups.json": "70bc17d11d4d9c2ec29c8404f5e60e66",
 "Experiment/EN-T3-01/run.py": "88ad33959014aff7855643001712af41",
 "Experiment/CLEAN-EN-01/submission_clean_en01_dev.zip": "ae902d1c7e87783bf88e345b459e39f2",
 "Experiment/CLEANTUNE-01/ct_round2.json": "21fa1fb37978e72c910f24ab6d0c364d",
 "lab_entry.py": "969ad42a47449a3a239cd9323a455f5a"
}
DATA_DIRS = ["kaggle/output/feats_av", "kaggle/output/feats_eval", "kaggle/output/feats_models", "Experiment/T3ENC-01/feats_t3enc", "Experiment/T3ENC-01/feats_vox2t3", "Experiment/T3ENC-01/feats_vox2lid", "Experiment/VOX2-01/feats_vox2"]
OPTIONAL_DIRS = {"Experiment/SCALE-VB/feats_vb": ["SCALE-VB", "VBSEL-01", "T3RECIPE-01", "T3REPRO-01", "T3RECIPE-02", "DSEL-02", "CCOPT-01"], "kaggle/output/feats_e6144": ["ENBASE-01"], "Experiment/CAFUSE-EN/feats_av_full": ["CLEANTUNE-01"]}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))) as z:
    for f in z.namelist():
        b = z.read(f); assert hashlib.md5(b).hexdigest() == CODE_MD5[f], f
        (LAB / f).parent.mkdir(parents=True, exist_ok=True); (LAB / f).write_bytes(b)


def link(target, path):
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(target, path, target_is_directory=True)
    except OSError:                                                   # Windows without symlink rights (local smoke): junction
        subprocess.run(["cmd", "/c", "mklink", "/J", str(path), str(target)], check=True, capture_output=True)


for d in DATA_DIRS + list(OPTIONAL_DIRS):
    src = [r / d for r in ROOTS if (r / d).is_dir()]
    if not src:
        assert d not in DATA_DIRS and EXP not in OPTIONAL_DIRS.get(d, []), f"no attached lab bundle has {d} (needed by {EXP})"
        continue
    link(src[0], LAB / d)
for d, src in EXTRA_LINKS.items():                                   # features extracted above (FLAG_52 / 53)
    link(Path(src), LAB / d)
SAFE_SRC = E2E_DIR if (E2E_DIR / "safe04").is_dir() else next(d for d, m in [(m.parent, None) for m in IN.rglob("e2e_manifest.json")]
                                                              if (d / "safe04").is_dir())
for f in sorted((SAFE_SRC / "safe04").glob("*")):                     # reference tables of the gate (streams, rel, v11's T3, ...)
    shutil.copy(f, SP / "out/safe04" / f.name)
assert (LAB / "Experiment" / EXP / "run.py").exists(), f"{EXP} is not embedded in this notebook"
for f, h in CODE_MD5.items():
    print(f"  {h}  {f}")
ENVV = dict(os.environ, SAFE04_SCRATCH=str(SP), FLAG_LOCAL="1", FLAG_IN=str(eda[0].parent.parent), FLAG_OUT=str(SP / "out"),
            VOX_LID=str(LAB / "Experiment/T3ENC-01/feats_vox2lid/lid_summary.json"), E2E_DIR=str(E2E_DIR), E2E_SPLIT=SPLIT,
            E2E_OUT=str(OUTD), PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
log(f"lab ready at {LAB}: {len(CODE_MD5)} code files; data: {[d for d in DATA_DIRS + list(OPTIONAL_DIRS) if (LAB / d).exists()]}; "
    f"{len(list((SP / 'out/safe04').glob('*')))} SAFE-04 reference tables")

## 2. Resume + the 20 T3S job tables + the RFSTUNE tables

In [ ]:
print("[cell] resume", flush=True)

# resume: a previous version's e2e_out/ (same experiment) -> finished jobs and their files are copied back and skipped
PROG = OUTD / "progress.json"
prog = json.loads(PROG.read_text()) if PROG.exists() else {}
if not prog.get("done"):
    prevs = [] if LOCAL else [(q, json.loads(q.read_text())) for q in IN.rglob("e2e_out/progress.json")]
    prevs = [(q, p) for q, p in prevs if p.get("exp") == EXP and p.get("done")]
    # an output made by DIFFERENT code of this experiment (e.g. FLAG_51 before / after the SUB21a amendment of 11/10) is not
    # resumed: its finished folds would be mixed with folds of the new recipe
    changed = lambda p: [f for f in EXP_CODE if p.get("code_md5", {}).get(f) != CODE_MD5.get(f)]
    if os.environ.get("E2E_ALLOW_CODE_CHANGE") != "1":
        for q, p in prevs:
            if changed(p):
                log(f"!! NOT resuming from {q.parent}: made by different {EXP} code {changed(p)}; its folds would mix two "
                    "versions. This run starts fresh (E2E_ALLOW_CODE_CHANGE=1 overrides).")
        prevs = [(q, p) for q, p in prevs if not changed(p)]
    prevs.sort(key=lambda t: t[1].get("split") != SPLIT)           # the same split first
    for prev, p in prevs[:1]:
        same = p.get("split") == SPLIT
        for f in prev.parent.glob("*"):
            # another split (dev run -> Evaluation run): only the fold tables and the trained final models / heads move over;
            # the final job then reloads them and scores the new test files
            keep = same or f.name.endswith((".parquet", ".pt", "_selection.json"))
            if f.is_file() and f.name != "progress.json" and keep:
                shutil.copy(f, OUTD / f.name)
        for f in prev.parent.glob(f"{TAG}_fold*.parquet"):
            shutil.copy(f, SP / "out/safe04" / f.name)
        p = dict(p, split=SPLIT, done=[j for j in p["done"] if same or j.startswith("fold")])
        prog = p; log("resume from", prev.parent, f"(split {'same' if same else 'other'})", "jobs done", p["done"])
prog = prog if prog.get("exp") == EXP else {"exp": EXP, "split": SPLIT, "done": [], "seconds": {}, "files": {}}
prog["code_md5"] = CODE_MD5
PROG.write_text(json.dumps(prog, indent=1))

In [ ]:
print("[cell] gather T3S job tables", flush=True)

# job tables ct01r2<face>_fold<s> of the OTHER runs (attached outputs / tables dataset) + FACELORA-03's B3 reference tables
import pandas as pd
for f in sorted(IN.rglob(f"{TAG}?*_fold[1-5].parquet")):
    part = f.stem[len(TAG):].split("_fold")[0]
    j = f"fold{f.stem[-1]}{part}"
    if part in ("farl", "clip", "b3s", "c5s") and j not in prog["done"] and not (SP / "out/safe04" / f.name).exists():
        shutil.copy(f, SP / "out/safe04" / f.name); shutil.copy(f, OUTD / f.name)
        prog["done"].append(j); prog["files"][f.name] = hashlib.md5(f.read_bytes()).hexdigest()
        log(f"gathered {f.name} from {f.parent}")
PROG.write_text(json.dumps(prog, indent=1))
B3T = sorted(IN.rglob("facelora03_B3_fold*.parquet"))
for f in B3T:
    shutil.copy(f, SP / "out/safe04" / f.name)
log(f"jobs done {sorted(j for j in prog['done'] if j.startswith('fold'))}; FACELORA-03 B3 reference tables {len(B3T)}")

In [ ]:
print("[cell] smoke stand-ins (SMOKE only)", flush=True)

# smoke only: group tables not trained here = noisy copies of FACELORA-03's B3 tables (wiring of gather / merge / report)
def standin(tag, arms, seed):
    import numpy as np
    for s in range(1, 6):
        f = SP / "out/safe04" / f"{tag}_fold{s}.parquet"
        if not f.exists():
            d = pd.read_parquet(SP / "out/safe04" / f"facelora03_B3_fold{s}.parquet")
            rng = np.random.RandomState(100 * seed + s)
            for a in arms:
                d[f"tr{a}"] = d["trB3"].values + 0.05 * rng.randn(len(d))
            d.drop(columns=[c for c in d.columns if c.startswith("trB3")]).to_parquet(f)
            prog["done"] = sorted(set(prog["done"]) | {f"{tag}_fold{s}"})
            log(f"SMOKE: {tag} fold {s} = B3 stand-in")

if SMOKE:                                                          # every (face, fold) table not trained here
    for i, (p, arms) in enumerate({'farl': ('CA', 'T3S_farl_sim', 'T3S_farl_r293', 'T3S_farl_ecapa', 'T3S_farl_cat4', 'T3S_farl_wsv', 'T3S_farl_rdn'), 'clip': ('T3S_clip_sim', 'T3S_clip_r293', 'T3S_clip_ecapa', 'T3S_clip_cat4', 'T3S_clip_wsv', 'T3S_clip_rdn'), 'b3s': ('T3S_b3s_sim', 'T3S_b3s_r293', 'T3S_b3s_ecapa', 'T3S_b3s_cat4', 'T3S_b3s_wsv', 'T3S_b3s_rdn'), 'c5s': ('T3S_c5s_sim', 'T3S_c5s_r293', 'T3S_c5s_ecapa', 'T3S_c5s_cat4', 'T3S_c5s_wsv', 'T3S_c5s_rdn')}.items()):
        standin(f"{TAG}{p}", arms, i)

In [ ]:
print("[cell] RFSTUNE tables", flush=True)

# RFSTUNE-01 tables: rft<arm>lab_fold<s> of the FLAG_173 runs -> <SP>/out/safe04
import shutil
ARMS = ['a0', 'ALL', 'c5', 'F1', 'farl', 'ALLlo', 'pel', 'F1lo', 'rl', 'ALLhi', 'sf', 'F1hi', 'ibv', 'c5rl', 'c5x']
got = {}
for f in sorted(IN.rglob("rfx*lab_fold[1-5].parquet")):
    if not (SP / "out/safe04" / f.name).exists():
        shutil.copy(f, SP / "out/safe04" / f.name); got[f.name] = str(f.parent)
need = [f"rfx{a}lab_fold{s}.parquet" for a in ARMS for s in range(1, 6)]
miss = [n for n in need if not (SP / "out/safe04" / n).exists()]
if SMOKE and miss:                                                 # smoke only: noisy copies of FACELORA-03's B3 tables (wiring), 3 arms
    import numpy as np
    for n in [n for n in miss if n.startswith(tuple(f"rfx{a}lab" for a in ARMS[:15]))]:
        a, s = n[3:].split("lab_fold")[0], int(n[-9])
        d = pd.read_parquet(SP / "out/safe04" / f"facelora03_B3_fold{s}.parquet"); rng = np.random.RandomState(s + len(a))
        o = d[["fold", "data", "protocol", "y", "fi", "vi"]].copy()
        for c in ("t3", "nf", "nv", "ib", "age", "gender_agree"):
            o[f"rfx{a}_{c}"] = d["trB3"].values + 0.3 * rng.randn(len(d))
            if c in ("ib", "age"): o.loc[o.data != "v4", f"rfx{a}_{c}"] = np.nan
        o.to_parquet(SP / "out/safe04" / n); log(f"SMOKE: {n} = B3 stand-in")
    miss = []
log(f"RFSTUNE tables copied {len(got)}; missing {miss}")
assert not miss, "attach the outputs of the two FLAG_173 runs (rft<arm>lab_fold<s>.parquet)"

## 3. RFSTUNE-01 report

In [ ]:
print("[cell] RFSTUNE-01 report", flush=True)

# RFSTUNE-01 analysis (README sections 5-6) -> rfstune_report.txt / rfstune_summary.json
import base64, gzip, shutil, subprocess, sys
for d_, blob in (("EDA-3FAM-01", "H4sIAAAAAAAC/91c65LbNpb+7yq/A4ZTM03GEi2p3c6MbLmq425PMuPb9sXJRqNiURQkMS2RCkmpu6Mor7PvsU+23zkASFAXpzOT3R/blVgkCAIHB+d+Dug4zvnZafP4zem7Zqst3Ivz07N35yKTkzgvZCZH4uRpuy2GcpxmUoTJvUiW86HMvK54087FU3Hx5hL/Xh1fijQRxVSKPJxLcXn65rzZeiYWYZzlwj0R43Q2yhtiscwmcuT5jx8J/C3uiynemoXDQCZFdu8v7oUNTSYXaVYI/ef+I5xMZlK8/njdEG/env4taH/ZfiESKUc5z4xxxCgsQvGEb4twOJM5QJ+ltyJOxMvLj6+epsviaR6OZeuZ9/jRFffoiqhotbPOy3EYyVcBQfoyfyVcrAlANESnJX5Ih3o4ryFup61O1evbr7+5/Hh+0Wx1xFzS4rqiyF6fAif08635/QCgjoZHIkoXMY3y+FE2pnWbcZ6KbJzc2i0uMNt8e/pVudaW54v3aTGNkwkwE2LRI7nCi6+/8sW3WVxgrXIUHo/DeaAQ5xd3RaNsy5fzeQgc/5CnCaY3zWk69hdh9uNSFkDlXSGK1MKe7zjO40fxnPdB/cziob8s4llD8Egixbbm9/iniOfy8aNxls6x7cUU/fQb4iNuy1FAP9jmMBfJAvQQJiNc4r/F6PGjx4++Pr84Fz1+wQ2CcTyTQeD5mczT2Uq6HkEKSnn86Py7j2foR93LNgDh08R+nOQyK9wW4Coyl7s+Fc5559zxvBfiV7ppykXXEmLZkQGtBlCe6wXiDQnkTcJCmkVePtfPeBOXefXgIzZBZg2FUlrl1enfALyjtwAI/sf5f6Kh79DeOw3hEBXT7yJLizRKZ3R9T/+MY/p3FTuDx48+XF/hpcuPBHVF1hjt6jt60EeXN6evzy8b4tOHb/CLNtcZhxmPFs3iBf0Oj3O+PckdkLbr5PGc7rPOX4/pV0bhgiGJwuIZ/d7mK34+ShyPxr+kqdLcl8kqztLEn8jCdd68Pf4+uHz34R/AuOhhpW1HHPr7o8jn6Y0Ubl6AFprgU81oXZGk4vTt2yy8bYhJBi6/JxGTJpKFCXZtHs5mYpimBXYjXOSPH73/CsCAXeOxYMjkDLtw3GoRzh8/GskxhNO9+0XodZX8yWkXhOP/kMaJS0Rw52HsTNyRuAhBK8CkHy4WMhm5OW4XWZwULkh9PFvm095VtpReNXYQFW458kJGGLzOMT61BkQjirJnKbAap4nrREW2TIBXQ4Sv356fvr+6fg9KbD/FIwhGQjeNPN8ddp6OlhiOB6YpXPqHKB0//iwNRzLz5R2mVh3duR4qk8UyS8RcLeGPovlv/4m2zxvEWygWMtNblaW3uYimMrqRI4UslpXu6yuDsNdXPjcFxXHueuJf+PtjXVCL5ist12sSXQyzMAEkOSkJCGaNCtr1nHYdTwEXpP5zAxn9EZEvRj5J3UALS5fY76kYO9Yk63xjZKnZLjP4IpNAg0xoDtd1SIdgv9cOweB0wVdT/IKx6H5oGobOhpmSVQV1N7esKVSDDSb9ffwMqGtAsfkcpPQX5iQWxUwm7kfmXrq68qD7RwIM57of+zcDP8yL+4UkpvH8VThbAqHoerX/kefTi4q3bggDkHdYiQaIaUQAoi7TiVMHh96J6J2PPgThcp7kWwumPzB8BHFR6LH39OBd7GMDWEJFDRF5A1I0/WiggbR22y/SXdxBaB/EHYmVsaP0v+CeMCoUtGLdj6pFXJlFbIM82Dje78iHUzkD9+WK1yBAgjCb5+5ZQ1x2GsSX0NwQY9Ro6AeK/vVp88315bmrlFvuwR6ADTiy+FiZbw02D+fYchgPQM6U9u86Gy3N9ddxMoqF2xPnPmlIj3jxrH/0EvO9+vmlevnV0UAZF2aTE9iNpIpZ9gIwHxYNYLbpu2hhy8jM8Okfd4vF1LgNWvuIBrns+F+fvz3bMw79YR8wGPaV5t38vFZvb5wXgBTPiDiShZ+EyS45sqDIIYHlyD3zCTP+MolBD+4ONzLUEmOd9XVXYhOg9s8C96Tm/TiH8iGYPaIJBp71FmuXwe5wi5Qhx+p4N9xq5D/okX/ZGXlQbnqfNZg38HYHPoOuiPqFhGE0kncNobGAic5kHk8Sl2aGFcO75Pl5BI8AtNKA0G9ZwyleELSHgpHbfdnubMSaxAje22jbKRf9tbWVoond7fqt8SYfOJZSDUcjslyJdBWBgPOKkmh521mkOiBFkp9Ef8EStFjeTIkYHXtjzvrWtm8cWiRPQc0L08ZCm4ZWE1YQRfMFQRM2xNDinVCs4Gp0LV6ZyxDWygr3ExHJ2QzcMhbn5xfukBZLF6En3FeCaDqEl1LA3YLSpv1qwpxDazUCGeRMGTOop2U4kUe5GrNiIVJRrPkIuPVdV7izcD4chWLSEHc93E/6d0bUWVYOtmodbn5W2MP1UF17G68a9qoPsYXZfOwcDL1RfhvDPHfExDH0uSDb8qpfvT/AEvlejz3w/EmWLhfDe/eKidXzaXWGhad4f73Z2dOHbOZV599aOG22WThdY+H+KEsXSWiLl2k/IyrBCmAJjmFSFa571elXr/F6O/1qTHvBHcaeWbH+gf47rtthcIemFZmN54W7MAvVPQgNylgdO+O14mRw1qT7xO+MN44STjCWmWyN3PNg6YAff1bktF7o+dU7wBgN4y4ECBE8DTbwvM3TE6cCJCOTkWAD60+znn+8BRQM8JlrRhWvetSLjYXaoPTgGTfPSSjp/SDo8KDpdywGu4VgyWgzIYCwGqZzi9OI+OKkGc1gqIjLhYScwbqI71zFeIxsw39j5czepoJfkOSnTmGGVbzFkQmGTHEp6SpIzzcZBETFXmEEmyFRSuEnmaW565JAIxAxV3lJ29qq6NjNS3B4U876Z4qTWDSrlQ1KQqn8P3oCErLI/II8uglL6Il/T0rkXiFo4DO65hKhlFHPyTVCnNIsIzjumeLh6bZtjYI1iSdGzQWweeCau+5FvzUAyVz02wMPCr0DmZRQt3adWEd+iSWXBnoqEAtgzdHT+6YMHb6zdhfUEUgAPOEe1r7C9mvO5ErOBB4DZti8E+xLwRsIcSlS7DH8JUQ/FmCBgvaWfA171wWjsto1gAFcQQKQLtMIwVb49zWbT3n0NzNgLgEdF1kc5cZ5z9IoiJbZSuquC/Lj+Z8pCwT91MVukIcYk3cGhlI4DbMJUTtuwiEIpk1KbpGR+Mc4nldnf/S6nSLi5vLuwgPJxUvM0o8HdMl8hevf00js+DrCphmdr93KI8Na2Kd9UXlr6pFlRAoy8uC6RzetjgsyP2fpHuinRoaeqW4coHBhHutmg2M5C/JbKRcnFGK5OFEPhxnpZWVlWRb0Wc2CpqmhnpVCKjI4eIGjXJUI/ZbYCSdQMYiOntNyb4aZevIMqjzTezE+bnAk7pjjMdkY/ijHO8ZBMjYXKwdb0mc3rHyMS9OBLlfOwPYMMIsV1+wqY+gMshu0FhKzKCG01stVkr2yNbeNzI1jua2Mlb5yJBt1R/Lb8uKDQzyN1fG/tLwnannxUK8KRgX3GWb9bnvQ3bXnIpKW6CXWEbybdbvVEl8Q0cb5OE6ga2A/lb6UUXJdvz3eiD8J1WPLs3sCDS/c1TMVHUiT2b0Y3iOmSbam5ygHic2AbTA9ZSA7jreNBmuB3d8aPXh3+klBQsREBhgUxMUbMZYh2FMaf04pDlfxCnyjcAi0eLZtGQ3Y3AcuEOSZwY5Q9oKSCIwWG1FmCSy4iJtGMVgO0RAKhASq1S1pHgxWZIgL95hO+TK57fF6G/t83i++WI8VW4yDNSyErrCdUXCY4qHFLFaMAplPHDFW5u+YOnEIcfMro6+C9eoBo3d49JUafUWdVGTSGJvpDRDwy7k/D1cBif8JOVj2HiuVRy8qzJS+3cO2mEOM4Yy0hlaO7tsLYyMo31YgwG6MAmAb8kilAXZ2eNs5/I2OITuFNZdwJwxh+2RYOE1KdDHLXMv3+wUD/RmI2+PhKWR56FTIQSm5tTOnB1MefqCYzq0FBxSGvV0sIm4ej6SAqs7uBQW6sXDaY8Yi8GnkM8LyEO3juCA+cRHMhYK90XAMw5wQcPkc7Hruf2q3g6tj9eQ9WTprBzqBOV7OJizEjgO+3BBNnfvvP1ycv1VSIEeMxqvBxw85mUWpq4b46uo1qPg/PErMhGQ5plPSOsYNBVQrXo1KXwEVURYvinhlciCaQaND6ojsObPNiuTrPhIZUP/i28mklHGUvtFekmpQJoA4bVtkrbsWrLx+TSMogWopsL2qjvtDzpQXya26BF+UF1VT5tREhRMmH0/aHRCY5HTDdwjo801L7265yaR7+jVhtSWD7A5K3myLkYG9fAvLFFOyom3vL3RHoM52HaOa6xj1InIdo7rrqMfbVN6dstW1woMBLWE9BqTMVHqgnxeWJeD8MwGv17aMTVvXOCs6Uer9M3Gw3NO2f+Urp7PjUVASNAv71zU0cdqG9iyX0FdhH064rJ6RjfLwJbk1U6embTXBsGf8wEVbCwaAOvhB+t3lwTnZSWMLeYcsM7FdJr2uWfPq2efWTHzQd07bHLYBglTXY+7KutPbZpCOdijIT5IzzrhgnggpHERcMLTqHUX/DtNYLHKQL3aYwQi1OsfUWYTvoijcxy7GaFWgWwSPhr0ErxydKF7c+3r5/jSGUZdF03tjjM/i5AZbA5NUd2koCZkjBZoXO+PkSKXChPVHMZF0VCY98x+XkP+AaG5FeOfam9dJEJLsk4hTGgldQ0rW9OQFCbwqGhCZWEDVYw55jj5RBJv1+WDbZlXM1hF7IwbgjmS+0fCU/KfLFLi1YUIGmhsv5qyMac4Gzzz4HJ2y9qYFVJggG5DyruwIYjTjj8Ivfw9JQAqbUmR57w18Agm18T1e1/vhns2JgJQJbqds0gzpPd7/CO4h4zkm/Fqb5n5ve/3Ug8nC9aEcfaj8KqRj/qiSoldSgPs9bE3KSus9dgBaNLNCdrVcDeiFRv8J66Qtw1DQvBGCh7KAlxkuZwX7NxQT6SNKptOr0VZAuvQ6FAgcCCC0ZdNUvMTMBXwQ2hQKbOmYmNPQFyuv0gxG587KiFND3Mj7HhwwSmMAn7lUQsvbE8MeAbQsnSDIofAMv4d/N/YOsEQaO6edgCiK5RI2d59cstJVyZzl9MTZwvzFhF/fkmFnbXI5TEkO8JmnywzMBoodxbSGuEAAgnATTkCYkKmoQHFJhpF6gFVjVakgG4TocXqbUJ8oRd0NwjDa+yV6U+qCwttdoVyXCxV3oiZjR46CdPhDAz/Mf5m7JS8pxGE1KokJzNfalPDc31XNgpUR6rhwAfPJiKwy9gx5fgqK+K0TdvfLBgZJ+YdxslgWujsaa73pXr+kOtO/HH+p5/RYhpxVhVH//V9gFsK0+/q0AQx+64l15h4R+EcNccTLP4Lfe0yxVdONF1XvyE17OjJSPGG5yogp197jHjtvknLaA8oH3REMK+MZVw7xC0QFFGPSV53dV/WLdWzQzstZEQblbog1IxHRZAZHPWXE05M4UQ8oqLqHZMFOeov9JcRA5laRDaXmz5SaZ/7heXqKrTT5HXuKBMtWkszUqEft6d/G/nywIGT01jddHdivyPCGo/NVvtp9uB1g2KA0A8gw4HRKjaPJbxpxnnPbOqyeCFdnZMVrwRFBnY99UuZhj00m1jN44/fIgUJ1Hhxy9rOeGLsGJqEDcMbkqWvIdFNyazXCwqdmMsR1Iy1msIVFmuFiZ4pyyTQC9bg63KXEk+57oOvByVEZtLtEFebCi6p06FAHij5RqAshiVq1EGKCe6Z6Qpu/Fzh6oMF/woRxqNeHsls2Bq3s6aboad9KnxCFHXpJ0Z4ZnI3OQ2i8tbtlB7tlzmBTURNYE26FdBHroTdNIGn71ZrrthM/8g6OZ0JHB8ZTjw9GjD5f2qAnrVLIzrVDkVviDcMKAxgztR7HO10ML+x0ZVfY7ktsZFQkKvvYo9gaSfcyY2qY1LR6RMVZJf8Y5mhUVF+R92Bf9nev2wc9Eulk7K7/B2B/g0+L3rZTyx4sBq5ybsqJ1R6d5cJeHXToWOoFKJNkgX/1ALfuGMbLfDGTcxR/UtEqlSNbRhKF7kyUtkmwQ6SfgULOjnfccXskYI4GeiK+I8MEl40yfa9z9d9xcY0lcY+3ktchB+RvKu3BMhlS5qaWrSDiRwKdad7a8Ktq4+0oPCVgydMpiw+Yroggj/thqSMnPTtFvbBTzahbhmHY2832NrhmpzfdY/6uQ127oZLRG9jaZXnCerqlq0+PeevCHfyedbasJy7h5LghoUjFEVU0joooyFwFgcUIF2DgEv1mslFnG90wS7VDadB4rWpZLW6xWq40u3GD5iLd4n0O4UMge6RT/E3k/Z3fBetU0AIfc9Qr0+ze3o2ACSUwaffQXpCFtT76eHp5eUSkVo2m1NrROIxnR9vWVUetobOzY8dbOwb/ORvpXdndkePDO6LF2hb2eZ8egO3j/1fYVvwxOt6PKMewNGHqutQY6jYxRPyrODNrcMmzQzm/94DV2NlEitG4PBYBtnaum8qQtCHc7Cmpm6WAJlblYlSSjdQGii2pOMDUAayr6qAuBbHGgSq8Ccu2CK5ZwCWv5rFuojAR1aojN4Prr3rvv9qza6+/+euJ1v9VUbgK8rgTU0WCorNZWpa+TGN1qSvObO3yTLhxHHum8ByouQ3BArmckd9D5x60MR6rtAFQWVZ+0UGYdEF6kPMbK5yfudrVOs8Ojo2pkwSjcQgHuks2aVzeLX3G5plKZZA/pyNWkyatkILJVJ0nF+TttuDulmqKMFmLcO6GLA85NFt+z05k0xSBxYhdcak+y4MGn0DQIgLlBRMu+yiyJKAsK9UR6gtTbmrnnKNa4eFnXirLEPl2uxaxoNiBfgd1yTxYkPdyO9qkyy92srR9GCPuxA/yPXUj2nIKdFlPUWybThx/DbaL5bztavMDuUJEM+teyYltChTZmDayqCJ5B4ahMQquCh3A/5+muUz0uZCyCHaZqVSmWrHaIiu7qEfkaqX+kidc8oIBAgZf0uDFYLu8tODko3mb4lwFepCB71UwY4jBVgFvUBXg70Qkh5LTOu/LJNdOfXgtAE7EbofA1eoHe6pz5f8pAijxoDYCZl80OIwPK1TIS49zXrsggwl1QdSGZH3X4IUq2uXW6+ZVrobBRjfNWzoStgcZCjYTk+XuLfJ6FJ3o1+uvEX3uGWkIfryxdBXJBpOZJqXFE1XPZaBKXg9sgCLifSluG8n5NpI9e4JxO/8dJzDRBGsKCD6DOTVbUy3L0srqpIMu3OwyVujAgIKW1LJSIxNOCq757S4pKOWUrHnYrlJeFOfVLWYipcy87ZCdVjPszegiUEPwE8q7GW1IptIdWU1VkSw9p2rQuqH/LFBDVjYZb2/PKABA1usrI+2ujKOZ8QZ2dlDYOo6tS9ZynF9ia1G4MoymcKlMJMwU5xRTJATUK17XwDdLxykHv5o70a+deFOTWj4Tb0IPuvhsfKtJz3SIbH9ga/MrYQoYRQT0+oZCICvlMjZUoIPajaG1sRTHjXl40BQsDe1qcIsGhcZsU7h44n3WKqzbRCekdSnD7KoYpfHSSANjLbDiME1ZRhWFGfw6slZWaYzA72iZqdzrU7FC/Ui6YxCZ4bv1lLiKJZgqndqcL/h8MSBhZrjWJE7yjmxnfUsUoxHerOzC198Yovn0Dids31HW5cT/9O78CsF0XL2hKxQ+hiuq5lLnKXTN0B9Ufl1XNvYdLMzRFUGqgBMvNahAVlXUUvDE/fRO92uoZHTec6JUZpRL08P2TXLwzF/FdmqQSgU/vaNoBIp3SajzQBwKe6esA2W0NPSvVwJGJ7GB6j3AvXnn8x7sOatlwTHegeNNCQc27NODBvnMYgAHJzoVwD+WRVVImCY41yjphMwyM0MCcCq7bIN+EN7t0M+ATZxaDeL2G0gwJveu9sv6dUwNKkwFO5v4CyOKNrpRNfLwUL8/Qi02eCNETsVE+/u0TZ95rE714hIlH1TTY21SQGc66nPb53f6CgRHJgHcOYdUEq4nyOuSL+TQ6Yin+lSExmO4D5HIGk/CRbkVW2h8oSBRNaA1QNR7tJxQrVk34Ex9fdF2x7bVUa9d303jydSprX4ik+05F+NgDj3kn9CBZ5zJBxfOuA6Fb8aSb+0xFmn+oM3TDIEnJKXwlvpYAVOxY7WqGkDV7G1p01KMQVETbhEXpNdYxpGDSbpV6b4fgV6KX3IKns49IVinsFC+anUOqTi/6l2qXCVx6wGWfKWSUopsAbgmIr7SNTsKs3xB6LHtaXUeVRnGqgqD6i+2Le58OeRCVuyHPoVOTE/Xgx07EwBRNltN1WWVgauYEoS5qdMnGQE9qsJ9CfDA6UMt1nkfSNLvOe9GIVPyWyBREdTAkWm3D9j6Lv5RJyHafJyO2vIVS6bZioQLN5g+rQGJAuMBFr2ZLhKYqSoB9W6tEA9PtKTfcwhPIaeCSB0So9cUuKo0EZPHlDnvtgb23M72kdsDNrSahEaxXjagakd01xmarbRXSi+WbiledfYdTpxQvcHETEDOa4+P5aYFHdlbbZw9PsnVSZltmEzq6YY7nW64+3y6wd2bX7FyIaXFs0VoVye+nC8oc0zxrThZyt1ulHZBv8/WptWKXbZDV5PJTuyK4N0KXTnXD4hb9evIHByMYlXAdAiazm8Hh1D5vwGRkkDbrxnLP+FoK6n1CYVZURIedWvn5Obk5pIk/E27fzBjf+jvOsBJm2UeYOQgio37QVvboboTvpvGdIcVY94D3Tv1/ny7zy0vQ8WEk24nZ7R0X3ZaGxyWWit0dJ+PyKZmz23u46f7nH05+mIOGmjxuoW+S4AWQoNuuab7a3NzzHfH+vZn4ezHzthRdjB5PzBwD8Q6aX67R8fq0rHioZbDd6LOyDIhbDsFz8WUgpfao9kx6uuP4c5xKWZG302BA8fOXe0AF5n+fLwLypL8CMGneNQptph+yQQgC185FWrcFxTRaLfY7yjdX563rjc/Q322bJT2MT91UoeOdt2xJ8BH3LbiVuZ034NO8vEM/fiOPkQQ8GgcZ+Qrjo1YJ+QqIWrR4ILr/SxtWDuLp/zxHqz3ytqTPUnMub/faqufdXLVWcA9QVBjMsGKSJ6yPRH/JGtxDw2PzPr0v36udgRxR9oGdWKCKvW5B4808JMZzschpASX5I6+uEGCxO/gUBGhHB1hJXsaLXtyNHwIyfTcVB41u42KytY02xE14FsD5kASlykhPN/E10L+RLiWOX+zoNa5NJz9v9TLoH4Io+0aOYKBF4k07J/Vcod8clIvy37+c/l8y6r8O45QEr1qc4h6NXUuA2jL+WNbT1lwAAL3CHdUp0UUTHVazL7UgwWJ3YNou+yB7lYPfrnepV525hLuKZxFCXaCgqBjrCH7hXNPCBO0/Hbb2xIWz4Mf1GJYaFTVVauqpEpHOJgESQLTiSaCqcaZHi8J50LqzxS3KhlePquzsleGTbaF1ZcqcACnLWRj9FbCHyl2pdaXYhgnqNgNZ6okt8wgNcoD/uKnphprWFmzZP/DjpqH5F3xBVze0dELHF3FVw4wmJqPstxt0RXtsoTgy39NUHGcpR7R//VTxyiyHjxYUN1hAXdtNhGto8ctWzLZ52qtPu19feqDq6yMipi6d21UR4xcYpy7lrpEJhKt+mB5k5p1rtfqY+dvwy+VoGG6AvoD9tRq6eIy0p8FdMyZvzWg0g6AZWCyyCN86GkuD77YPvDiITGlSOKpIgjkYwnM/lEJ4NFA8+foqHyoINBPyvysfU4ZxZz4pU864cS0Lrp/m1L1SxxdSARaYdGnSekkX9/q7zKY2MALUaOd33B8raDTz1L7hnbwvB4qt6r00XcXNBL7ARg163VaLWTqcVAMib5+v7RtbZt2UFEZvkFRpuFoEYaC5tCHsHWxrbRBEL/VfctWV9ufObFRM/dHMooJOnzBLeFkrFvIX4PJCs4qmnAf/BkSzL3OAmVP872+1RnVelj32lQB1HWHFirqQ4xk2JUiqgovUMrUDka8EExFCJ1qEUgRORJJdq7cBIYrMdV3boNsnAd0FJgyDwx33+aTO68e8b+l3CO9eH2rTe7rQEW7DlZx1LTJl6qo6EsjyfX3mJx9XzWEsXJL30EMqPDBpRaE4uY4P0ud1McJkqLXpiplPofQo4glrJ4kSkdIpfecZTFu/sWsdnum6puK9XmgMvS5A3wtjj62QQ0HR73BwR4siD74RzkFlroU2eIDQauFDhOynaWjt2XwjD8IiNAY/67wy7Vjh4/GV8lnZUFS939W5Pegs06fPfq3Uy9QBlRtqLi8m0QXjl3kLq3fK892n4GHdr615ez5NKVN886eT4TSfhQyMV8OwvhBQC5zEHBgKgjovE4QmNMWdL6U3oOvrC6wCP7GARqUcdfV3zzgDyB4mz59NRLm6opkCwWH6GNGugmfMAHfqFMEetgBlSv+Dw/dAyJbVQAA"), ("RFSTUNE-01", "H4sIAAAAAAAC/61abXPbNhL+rl+BY+cuZELTkpz0rkrkGcWW20wdx2M5vd6pGg5NQhITiuSRlCyNov72e3YBvkh23E6mngkFgsBiX59dADEM4+ZidPvxanjU7gjzZjg4fz8UmZyFeSEzGYhO57jTEd1O72Vb3MlpkknhxRsRLxd3MrN6ArNFmiWfpF8kGSb6YSrxUyxjTE5ikSwz4UfSi0WOAdIWo8EF1nopVl4UBl4RJrHTEvhLN8Uc4yPvzpVxkW2cdCMarGUyTbJC6D/zZ282i6Q4u/5oi4vLwY9u558vX9NkAZqeeCFuT0biU3InCu8ukjk6smnxxssWp7TANImCN/mpCGPxZnR9epwsi+Pcm8r2S6s18hZSpF4xF14uhueDo5OLwXtw0BN+0e5k3WqyeXbrLGQ2k25xkvfERSeHiJn0FrRakZ0NLPqdS5Li6HLwtmQlmQq5ktlGgBthauZPLKf10WUGRV+cDY4uPo6G5uh7kPil03FvTxo0y69FBspqjmWLxhsoXN6Q8sdv0kxOw/Upd4NPW7jxlB6riYglTByQDojJ6TKXmcjTKCyc1mh4CRp6gD9PQl8S2zSOmYa+MJgUISB5lNxjpFiQkWdHvowiMRze0AQt0ZcE7QQyM4UEj0y85Om55YibJQxZel4OP4JLiO+tHnNxRDT8BA6RRKck0kzQArk47YsXbQcCofHy+JWiZou5jIIjsCc+ZsFSHIufwjgIacxR2+k64gqLh/EM3uQFuQjkCkPO3uIxhDsutTf+OwsL2OnN7eDHU1f5nVOsC1v35MvFwoN/fsrBTizXhSgSFkzZ12l1ugiZHnwogytBUTl50iIJSM5ZlNx5YN+8vhlevPvVFiBpi8HN+5Etzj5c3d58uLTF1Yd3o6EtrgfvbtANNbgYPLJEntQR0YUMMFmuA8O0XjMPgZx6y6jIW3cSZoG1ZDOI5Nrzi2gjzJycHKaBxrjpJwuwG0KismcagtsYLZjIMIxWuOD4Uz9ReOcsizCyBSnBFgnN2uBRhAvZmmbJgiMIw/QEcY3XkgawA7ENncSpjXFx4LGG0qDV+ml4M4Tn0WjTdYkH17WcTOZJtJKm5YBJgENr+Ov1OYbR6LILyzu0phPGcOTCbNsUjSaPPBbGsDs0LOjoD4ZpcMLQklnZlS7JQWCgJFNB7s68QpbSjb5XnxhbYJWq/9pWXtFq/Tz8DzgeG+Snhi0Mgin6BXYWiZ9E1N7QYxrScxUak9aHj7eYM7omzmqIMlq3v1L/eNK6GIyoleSOjFdhBuedycI0Li5P/uuO3n/4GTKLfl8YHaN19RYDu20RTgXPkhHYPGm3W9+JI/w1vdVseEzpT68bnnecLWNCZ4roLAzgg/C8hdVquDTWMgC4JEk2zSkZGC1yclaB16Z+r53RT97hZ5eeBbcLbgfcDrgdcTvidsrtlNv33L5X7cCYIDF8hxU+c244vluGUeBOI28GeAW/rT+IL+KZWTNL3vAGYB1PbMW0os+zegilIvOEubbFxmqGz2uiJ8Jc9wE/70PkEoX45VKtVguKRaBtzOcekI5SWk7rC8P5lISxST65tgBqmVgTQntwXVjd8dJUxoGZ4zXNwrgwEXPTaJnP+7fZUlqarhslXmBS8Noch+UKqfSxyH4IO9TrkveqcIsSn2GwOZ1nLx5OVZimJhMZkx4UZPhxiAeZOXIN8mqguVCUqDrIYrHQ3HIORTLVXDazqmmJb/j7jvz8IP8fnT5I348kbGYghqCRjDVCqz6CSnTnkF4G5jZ1SDvjuDdxOGOaRllVGNa4PWG7pWQ3RLBDkG9Oja2it3teDu0QdP1vKQvD2qlVaFpO0zIvhko6NqfBUiyK9DRwKHW5eqZJCHEspkZDtm2+qwlXk4m0x54ESWqa9Hf9BN2S6a23K9l+nD4rKSdIVbpj3KHWrYV6MRBeFJnm9fjzxPHyYpNK8nDLoaQL42Do7eOfLIcmqkD4TOwDRhGSe4ypOgRs9USW3OfGHk800aeJ1w5AdrmID4SnP0CiL+Kk0PQfDmDtj/0JZSb8aN5qwzhF8lBzQMHdVxRGcT812MvBOA3sKQ/b0nNn7IUJdelIKVO94rBot8EQZVyHHqaahYoVvQoBDBl4JwCyKgnW1axB7wrHDeswfNDvMj/HwgsCKnjQ8hcpCbYo8My4ZOM6AXyFcUiIkVcMOCpB4akjuubIL0C8wdLZ5XBwpZPNHk/NsKsQoqXNRcaiLFabSvuemlAFKqUcdhfu15WuRgSla/G3fnO3s21O3ClXQsVEJblKHrR61VTGUamUWgh9njHqAjiRSaETzBw6eeH5n9tdE6wMHXIJV3/VNjtXozi7m3CHppdAGO0hWwqnc2sHqApRQtqq5K1Y5mxg1Rhyzis5yziE55mWtbObPmYLXVSLLeQrXQ4hQhxvvZ4Y78UYwNg4CDtsJB52rYzJPtTsKmDzsCQzRos4qLAXudlAtypQ/YMIVWog3pBCvZkEw6SMDrz/OSpIJ8zZAaV5Xoem5dBmBOSdznQn/i7UCDg6rOeRf2hrKdNdJbHcW/JcxToTRFKMYs+cwn0LMybgj5l2czXLqkLn/eAXRiHO+rQPwZ7o5kJMpYdglj2hQEjtlcz9DaYywSxLlikbYWqoLR0Ua/RId1/VolJy8hmzfh86C2/FVQ/ABe5S6X9WzlQrVBb4szkVToxKPYKuhIZtE3gUeT7vIXPUgbrIaewk6/xwPp6RSpUC94zOPGkfPvTZfU9Avd0XegwHuXWIxLwSKg9/XKB24RVRTrhRZo66TrokFDkf/w46/4C2JlWQjlXsTCxSkYUhhZxYOpKdc5mHs1hTUltiN+A+sxHntlarmnbn5cQrb9+HjtrAK+v6dRXh1z5/7tCmtpKbMxJK+iAnLzINMaMNCQcD9p7kGwxoRk9v4Qwdy+QmCOZWBR/GbzH0RAYJVPCb+uxAnIEK6gu9xX8hWDmBLU4YDWnXZ9RwXuYD89wWezJvDRRRWJXlfYEqmaooA4p9/hzue6Ndd9z05EN8aJQ+j9QntLpOQbT41PjIRKggp5WpR62i7TX1FrzFoI8T4qicsb+A/nTz2KdWWWtxnqBVt35PmJG3uAs8jqI+3meEj/6OTjaMCgdqg4KPXY3iBh2HmBhqq8DHaYg6/Oj9Fht0VuWw89xhJPmBoeEL+2SZhb6bxNFG1fdj0J04GIs9QNdyuPBAtYQTDVM7yO2MEu/4VjkUvh240aReSperTFLBZc1w4xgn1Sc9Ja+zJ9cnBx0bUmYuH9MYFILfJB4OxEA8CH3UO61GPmSPlp4/55OoVa7ylzA/otpcyei1OBVUFtHHO1kgqyOo9ZmR9mgSCCG02/e7sbfvBypVoDoA+UntkDixmFO2hl+iINr3SW6pbPqaVhl7JD2LMOsrqVQaSbW+wdoJ/t0DPvsEQWZK3FsOeTt9nEPt/XkNomUqRH5+82ontsQFCjIzRVHwpT762s7LfM7WmBqrXPNF/ICxsozifTClwVJmqtRVZ2fyMBi/IntJpVJBSWHX2B8ov4iTMMd+MUqSzKh0k3ph1o9QgZk8kez1lLa8u5zL6oMx6Dat5siHassloI5ZEMyCqFnHOWPNdk9F6bZcuPfC6U53Ona/0Npf6GNzxR6NMKr6VJ0R7DGgHLdRapLYAOQ1lt5ULlq677p03gcq5GmsPO3BpRfTSQRvdPbX1hyt92y8eXwn+ISR15V1N7uDbd8D3gga17sjGvmXBICudPvjJokVz2XJVyRNOnmEqzJe1mzgDdWNT8bMQeLEcU2vTJEEKPok/K84/lb4rzW59ChpfSVVAUpC/3O+D1kZfTYNyj5I/6Qmd4kD7+plTsfeRqN+Ogd5iPNlmymjNAqxP1+EUU0D/Kf6Cj9YQq4JIVEILL0yhyh2MVA6YbBehFVXVWURBuF93O1NKsqLP6rsvknouhxckPM25W90b4mbnfryADPU2YLiHO7Dsm1L9pF2wjiGNWds8a10WF4Gg9c6MW3lePpM54VnE/5Umh27Ei9ilY5r/VVaq/SjYgvMNyCT9Nnnpw2kyAv3FUMiU6wLKtz9cGrLavVVh53AhKzysacwnlfWsQ+CzeiPEowOqdDtOndJUsCCOIvHqYCpz1r622oaV0o97O/l1DUU1bLLByS5xs6y66+qC7Do4+UOIYP22/7V23rtlHb8gWaVziNMYh3H7fOs75w8BP/yMomZeRIH8Hr27odXusoVJBhqHC8V422UlLlgOw9Vc0JHiwdnTsb22fVgNHpGStZ88kbz2cXg3eUzeI2+vGp/9fZKX1i1H+I/+4HO6WyOvwhi/fCHVyXCkl1pknqbh/RGuyUWpa9+mmcUgn2QIbKCOS0TNM1OCaEVQKpIgctzVKhvzybNNGqXkVMOUqHTHGLVqZZPZ9TJfe+bzooblx2Hd5DiPskCFLeVCVa0e6vNoG5bYAm+NtjuHrWVs0xx1yNhnpUaPzNKe+gOahulKXTfnItzbRbdR23q03bQveoN/a0nxGQ90x5OQQQp7mB6fRSiq2Ts8+bdRyoBLiQRlbTF2ncCk7ahuBexhB6GgMOoZrB1D6OtW6Xdg5PT5i0rdgz3dCfrEnWTepwAd4i5SYNsQYkoLvrYzurbqj6dINtCxn5C5usby2J69C/N7cE69f3u/iqoAPRdDI4x6dadOr5GU2mACZb/UYFIFRLgu22c0EIhOLXtOe3pLp+Q4C0YwnUJMl2X3dh1F14Yu66hIHlrKHLwbdXYjekG08tmKy7Rp3zGXnZZCO2OMmQ5b2Jarf8D6GOwm+QhAAA="), ("RFSTUNE-02", "H4sIAAAAAAAC/2VTbU/bMBD+nl9xyr44UkmTUGDiTapYQQgKpYVp0oQsh9itNSeOHKes/35nJw1s84dczvf47nl85zAMl9er55eH2UGSAVnOpt/mMzB8LRvLDS8gzcZpCsnx6SSBnAttOLBqB1Vb5txEp2B0WxWQAet+inSEp2ttLFzAkDkdm7aK6x28S7sBu+ENh5oZVnIs0sQB4Kp3dqMrUCynvLJm5+CfqPVJ+0Xu2HqtOFwtXkZwfT+9oenJ8RlYlivegBHtOTPlpcsltCrOm0vQYo87AlnB+WpxOdatHTdM8GQSBU8pcocDYAkQrMWZjUbwlIE43h7BGMTX7WEffd/Itw0URtd43oEOYTW7zzCKCUjOLWqCWsvKRmdQaYlahdLaYNh4UAzfmZIFsxL16krt4jAMA1l6gZ1RMo9bK9UIml0TCKNLvC+7we0eAAt0g4A2NX/Dm/77VOx2qTtFhVScKv3mi5HQiMa2FU/ScOQzENohaBQb3mi15SSKsTPYgd6g9o8JSUPndr0Mo2D+f+VSFy3m87UdC+K/eBHexkqzgpuY/0Z+HZTMMU+8WM6ub3+MYB4/T2+cuXp8eF4+3mMB5Nwi2z31zP2zJMRD0+V8hYCfzsXNIu2+xhnXNm+xb96ebCfeHm19BjHZ9vC6t03WWZWGrzhhXzDzLz9k47yVqqBCsXV6cujG0hXOkMDD4+1qhjiyr4oUvJrp7dIzIz0pJIiDQgZWg9+z+zfu8nyOD36vYvB7NYM/qEI/gI9FPgntoB+C974a4q+oAUeaYlO6Cx5UvAaBFEBphU+XUrjA9lBaMllRGp76gvO4e6kkCv4AbUi9ZVwEAAA="), ("ENPROG-01", "H4sIAAAAAAAC/61beXPbRrL/n59iFqlaAQkE85CchDJdJUu0N7WWraLoeF8xLCwIDinEIMAHgIcezf3s79c9g4OHZCVZlU3i6Onp6bt7hoZhdD/c9j6+O603hNnrXl7fdEUip0GayUSORaP5otEQIzmJEym86EFEi9lIJiKeiOw+SMU8iaeJN7PaYhIspehG0zBI70/jJJBRhvFyPZdJMMNNaou7y7fd0/qZWHphMPayII5EHIUPTk3gb/6Q3eNB6I1cQCcPzvxBlKQlch4nmeA/85/edBpKcXX7yRZv31++cxs//nRBIwWweuIH0W/did/jkci8UShTPEgmG2ln9mL7yktmr2mOSRyOX6WvRRCJV3e3r1/Ei+xF6k1k/cyqdRuY+eryzXuaWUZ+PJbJ6SiM/S8CCJlwW3Sb4ubXX7qfCQZrPNVwasVZKiaLFOsfPYiry9O3n+66GNEC2s99GqD5JFYymN5nxM23jVS0Re/tnV3rngHu/eWHdxXI03Qu/WAS+AILEP59HPgSGM+J0I+9HkH6cZJIRV3qiDtvJsXcy+6FlxLa/qcPXZYxTdTJiTLvXoI7vzYabr+Fiyy5urTsWs9dV0CyBMPdtWWLT/ycEPwgAGM54kMMLYimEI83TsVYLsULcfUGH13IeMG0OOJzEmSQgoxIWVwlSSdbZ3b+KF3MZh4k/nsKcMMwasGMpa2+wmDkLLIgtAW9t0X6AFXKoFO1SRLPeI0A0cDiFrf5eOgqlAjrj+Y24KIxLvFvPq7V/tHtdbEWgjZddxKE0nUtJ5FpHC6laTlzL4EW1rr/ur0GGEHnj2q1sZwIN4y9sRmByTZTYLVZiUlKgN8l3KGnLhGrJoIiMWeqw3n07HDoLB4vMIQHExqTPqwLnsghGmTiyDXQK0BzpjAlMlskkZiB2hsgVdQaySTNFpGsNwyoDq3shTBK1TDoNllEMDzDEjt/3xHTHdYms9vsFpYMnUi9B4zr/6tvi4+f8PHP7v/AKC/vcPnhTY1AX9rKDqsAGFVCgcAbB5A3DsHeOBr6xmH4G4dH3Dg85sZRo24cYO9fvsNYQ2mRUfuu6rhgJqnyUhL/EynZPyTxIoKimh3xhX3Ii9EiCMfuJPSmjR9b5HIuezd3LZu/1GfTqvU+fvpwfYepNmChNNpiYHh18BCfCX393+qev1Jvxt9Js9ngi/WSv/wwmPNFMJrQd6wB4xwwVoAx8AztWsl3TJcdTpfymLRJnxlfZ3w95usxX4d8HfL1nK/nfL3i65W6Hh/MtqjMptE1eM7Jy+U5f/+0bPH3j8sz/j5fMq7J2VKDz/V3qmkCJcNt7ar/nkQF9gF57f3HNx9xO/hjnBuSOXZ7/Tse+hQTjwz9/K73C5nyoO7UbVF3mvTROqdP/njJnz/ZouHUMdGHdzCRD/T5sf8Pnk9GrlxnYgqelXeRvl2e4QUmWyTjBYEAAQYBCzwlodFeYyaTqTSv+tpbXPUdfuJmrdTcM7nn/X0nmvX9aHf6WvhZvZE0ixjHPj/N4KRnae7lmYB7DzEbar3lO8R4MZ+sbWU7CIxK7x147xnoaxeqMo0zYonHIzyCVCMmwoTJwh1MjA0QbTfeNg+1DXKf/7uQmWHBX8FIgXBYIMTICDjJtNu16vq8NEUwVRN2eBY7R94uyGQTV2vfAHIr/tapeoINwW2N6mSA2p2HGDEA1iHWhZe1ytxs8jQNwUDEUYxsIp/QKBjHlCReBPk2bPGywq0+cM7HDoVIVzOhYFNFUJt0W/KoGHwgE6LiUCI5aCGM3Vf0d/sEGfvSOk7LnlBCGZm3FsmFrvoW0kO43TA0zdvBl6HjpdnDXJpQO8uhZADSAWj/+CvLoYEWr+ELrQE+37IrlAkiS4CuNnz4KjUOiKKRPo00jYw9VMSuNloa1iEvWCyDEru78bcGyf724KEmsJSmk8WH/EMk2j7COYQtc2KwnWMBBNgWBqzQuBCG83scRCZNuYWabg0lbluJsSppayesM5HbnQFHtGNXIYbaBY1DE+HTs8XIFr4Mw1SzB2kX8lZmsw0/5SE1X+L+3wzzb4qk3W7PHFnilC88S5gxhVcyA9G/EK9FHfzzUCpkMDuL0zjCS56iP+g7hMcJUixXzTqsLsjMBqMhMGcDb2g5U0Tp+ejBzByixnKIGtPS9E9mmTnXNOvRJOeck5NNEGVmaoGf0/YPTnOieQS/MSVWzAuOQgQTQ3xVa93M9SxqjC0YjTmnZVkO8lMM2L44NzQR8P4JZVvzKdK3aLpLziiOQ7zRCMXrjnDO2TbwsIKPXpzxc2CoAJ86zXytv6f7Sx0HfmYSbIczGXOCxA780MNhMS38XwVR2jmk32bhpp1BdeiSxzCLlsyfYT65ytNzJ5PVSbyUdTv0YSp97F5fltmlHHutSmKJd6ett5c3+5nlBY1ykDBy2odvHQpLRH4G2Aqmq/fdyw/HklTlpkn1O2Vo5Yd3CO9QZ9QL9K7rpJnnf6k3Tay2y1rl6rd6IdcKius/EzaqnvpZSHUQUhfEDP7SkU6GqeSrQX1YMXL4N23gG3KJ19DCLAm8MNWsh/8CS+XYVDpaxo1rpslZRAEcB6marTDQFECi4p4foyyOQ7EBWVu9eD8OaYGbdZudwnrrsvNTl+wB9eUS7o2mW+drKEM+govPVBCyw9BSOFZ/L6hcD3xymfTloK4JI0/rVDR3Ii9ileSX2sVbVpG03Fz+ym4cqoqif5HBDFCjCiSH0HLZJkoWs0iZpomKraJLat3sInjlE0PVp1glclcs5NElqRXHXzDqP11n5i057YdXhqAKZkzzkWqGgh3PTcegaCgzwziSQgc9E4YUej6cP8wyGEsh4VcfuD0A6c/DICsj7PVgSixVDNyRANOktWdfW3bFklFGp2Eo4qbWfgjjmVA8+oMMCQ3PiEzUDRPzrunMF2RH14P/AM/fwa1hYUgDpbVDi1hkASSTw9zanGuZBtNIY4okUq+xO+ZnZsUWbc1WNUxSNwFqi3SqiGUcr/I0H1ZXvi3iGLOEmytUkCG1hdy5jdF1ikbGwKAkl3Lz77+HivS0egyq2rJvEEo/MCdZnlyzM25W7ICmHADVbMmpwg6uEpVcDy+qsJd7wGy8Qwo+VXyFEzF+iyAztTx2NaZuw4groEFmqXiL8SwoxOoW57ozGJhROGUHHtKl4ea1LXb4z5gV3ILsZ7De5QJRNogGzbbiTqQyKiYSUVYtyOJ8nd4wtmHpSejZouImiBRvPKYGD1EyMT4xt6hiIrnREyUarUiraLcaWRFCrtzaf6o46rbaaFqtgHSl21ZmA2nGykL7yqsL89MJv+IScDEXWSxS3wtlaZBEz8T4zNmACpsNBMLvgcFq11vjLZK4VTSISMSrHXtNFOOQIhHf7iniUG1Y3KBrNg72U9Nr8uHR9usmUdnoCjPRM5Cun5ULUG96SqHUW8ZFPSUWLHOYxZmzvSoj/aJ39AUajpm5ihQbKDBzlkcS3PhtYYbebDT22FV2cD8lqv3tV1pr4ezLsMEEbXUufB+vCtq0PrBBaE0ojGPgVxP6T1r9PrH6aT3c00cG7dPY36o85cUfOBjuP/yw62dKsBwR8+dZY3u7Y6trZV5S2uZQD++LfEhNemOVxYGBfJqT6TzvnqhswWr/FlGh0C8SYoNSZ0MnxGhuorPuu9RB7/SThbQGhHjoKD1tWg6XKiiy0KA18/nQaWW3SbZutHVmYejUggJohj6rIWXicpqOJ39u+pZFk3O6isC75978e+l/SQ2r2nVQkqZ+l60aURXbYB5zWYbm0fEKl4KByj7MvlJ4DU5lhX6itAul5ggRHQnAOudK/qeyOCGK0WKZCj2wLTBAfCVZjYPJBJ2FyJdfVZHEoXbN5ZTYjNvO2WQrzLFEHTQLIsgfXfss8YIIoKeUkFMSTJIAWV7dRR9LqgnZ6MfKjCelJ8TyVeMsObJ49tIzXcsxE01qTlE3Cu0nxGq+jdS9foCmPQOgtbWfPcyJAi4RlUGolmPVUFX5dgEKB9GMCX5OTQKec328B6NFUUIeF0gUB0i4MYiMsA4uCvLem2i2RWZ2vIDjp19pyq/0sjpRm8vC0iAGBuN3GT/LIJrUKmk962x1T6gtKL9TG0KpzChusKowixBVkEEsZZjXxHmOS20qNUbXyJoE2dgNeoWXKSWz5jCkfUzVLNbcNztqEWTTQbSQxUNdrOIjLoRcRuSqiJXiPA3xTRBSrmLuZFqZsldA93amLKB9L8H+IUWxsox+RZVxWUe/2q2ji6H3C9p7qw78E/V3gY22DNC1zunRe3EGxxv9jEtBQ01bfa+eqLekARQJSct2xGfmWCB2HqCaaCZp6zzO7QKLdZoEgrqAM+rKq791yqf5KvZtmlbxQ4ebTvu7iSU1ualtvParl1t0ONAv2XCzZWptf4sKZHgR5W+i3VdwK/Bw+ct4511P2YTCqyw2me6YLLnGDUjdVjqOsoGGUO5iyKt1qCUytWjnED6N7yK6ZY/GtzHuei7DVl1NMZn2R5hIhh18Vt1At8HWLxuH1l9u9raP7vZu5Hp7sOVbCR+5rTd3bT2aqbin0hlOZvbCHsbDzRm5nfe/ZeS2uKdeBhJvfzYvkm5CUTXPksVhjBEBhqCIQ+cqQ+WI0IaQZKoiPO1simGc4qF7mciJa5SY88c+dNc1tpZdQOhHdu3RHF2XtJV071q1CvPqlm3Nkdi1IysyDeyukAyNEUSOGd50Pryx9l3crnvbX/sR/3YMpOqQEiRJiXQ1Y1XrD2yzaWex47SUPu50BGtHcgrMgsil54AZcCRta3NBm+eruJfh+JRixeaebq9++flcDDZhXHQl7wN1OSRrwZ4suhOC5hWbk9vLu7sT4lai3c7J28tf3p9UZZ3Tosy4Yt+Ya9+uaYL8lMLBDGDFI1PIZjUrUCYJ9lDN0wG//ODn87wBSZpHrFN39wHdDW29KrTY07RDfH/K7KUiUANL9zFF67nL1AXX95xCJSDlGrATkeAwKYod5ifaXTSVu2ju57flQY9Hy8+3dxd5Kb9CNqH2sFZxwXKVzsF/0GjUp1Sr5gnDKs3rCtRoF7q12BGP9Bb7e92iklahHDGltNT6X+l6A5tE1zhG8zYhJ0AF3177PrSGZUlAuA1rsEqHmkt5JGV7hsBD0FD7Q2U7Etg8kJIUdAorqhmtjjR5Cru1nix65oH/BRHBVhcxzp2MYaVj9hMbsqri/+5uHjO24o4lb2WYiqEUecHmv4sqb0DtM3hzkVM0SEl/AuzYjNeoFMzSjIL4m3PRyp8/WZzpyeLDyWK9RVN0DIcOsloXHQq5zou+Ang8VYiUBcXcRoRY0AiHU/5cP1cV1+7zcrlD0APGH0PxKI4jSGq7zh4aeycpjzLH5BUqt/ASFyJpkZUec9C5QyTcKZmhVjw4ZfTq29TbWQ7z/SO1P6KoKHrKYntRjtZq+Y3RcVYZbewSctd9v6IYcfIXHXTrmH/WXqtlFP6ZKcdyOs9brBJCihXcd563wMcc8vOdess6TMYq5+PauRctj8Xl7nTXlXKhvr+pcpigLTzVrd3vmHFvFKKm81pT/xseZKqm7BzJbvrPyG6e9EJzqYznTziihbfnGw5dAdb2HPRTvYf7Z6b4r3mbSk/n0OPIZzmb4yh2cfxBX3P2mK+hrhGpBGpPJcO/dXJ278r6ea5pWfToC9uTFb9CpVcxtEw1Hxs69Stj0VkhYrHfrVtd1qGXgluo5LJ/zVmdPeGszh5xVk8uP/dT0863l2vrNXbo66+7qzPrIBmsntHFuYm2UO2AUz/ECHE3l/AttM9Z6SKWneA8L2Tgco/Hp1Om1Sa53vCiQ4LR7Xmjic03OeeMdn/f6fF9K943o70GtWtR7i5wZ79osl+r/rY83y9l/ytNyN5MV1krOs9MBazv503Hw2YF0nadtfZmZQ54kAEqeqndqjJ90qliQLVVXU2Ok/uYDs7ZvFdk6UM7laMmXM9tCO2JQnsyHJxo9uFyclJh38mw7bTysyh50a85VLBGs8Sq5uhHVGhkcd9eSKitT8fqc8OaUjjLmzCm9Px7hsND0iJ0TKFPAXoVvOxck3gbHrq0t4u3209UJ9AAth60W3CN60GrvXN6zyNPTudGv9Ge8OvcySzb80WTo9Ld7yt/UDrnsg4HsY43nyNg8p4KxfLOWteIuzVd3pL060oHyyJO2e9T0FXwx2sX1XN6HE+1dNQivVQBpCyvaEFWNf5hQRVne+lQBuGqPTWtMNYTeq43ry9Vc7OySUAH5VM/mD/QSRiogD4gT74H+8KJTefJ2Q0lJa9tsoAicuagoInIwNlohxlgIVbS0SIGKrAchSpRcyRF3xLnJcfxzMFxI28RZi6em3UE0VFFI3OtdMvzlc069mArR0aVupzX99qe1NQCRgcFsZxi/9uETBR38u/dzQbCiFZqNtbE663MQTC0uJNPndkCgFe1B3F4znBUaCsGkj/zYzk5QF9wqYJvAGIbwyNtOiBCse7jlwj4KYE5gvkPmrR9/fOPznmVxTjmSa3r3DcguIRk6eQJlItQ+3scgHWPXDWr8xGRlGgrkL+IsS2P3w1AzzQ4evF1DY2zjln4cKSNrDis0uy2uFXag7GbRLd7Teo2Qn29ucrNOR9/ouMFqRWxkhRzg49K55gWvNOIOh8YozJ5iDq56BPdAkpU96nakVJKHMZWtTGlHt4Hlu5PYdocw33ewcLc4E6Hvh91GLS+zuVhzFE/leoY7ACM3QbTuWownaucYu+4afVXM/AKK/qVjZvhVLrJv6MZ48cv2GsGkC0IN+bA8RFtah06e2ur3zYR5cYim5z+pNm3N0/5g53dWRCbdCzkY3x0oJIePYZVaQajzH/NRcgypIqDTeVgIR0DrdfbTn2yTYcGnUaEzrkutaRdl8OM686wmeq6OtjgRxmMDi1odbEd0K9VvGS6HDSGuU/MH5GlNrTy6nFD06r9PxDhAycINwAA"), ("RISKSIM-01", "H4sIAAAAAAAC/41Ze3PiRhL/n08xp1TdSonQgu3bBw65sg3OpbK7cRmc2yqFUsloBNrV6yRhQzjy2e/XPSOQsOOLywXSTE+/u6e7MQzj9qfJz5OfPnZ7fWHeji9GH8eikIuorGQhA9E/ed3viz/67we9nriXYVZI4acbka6Se1lYA/G4jOZL4RfzZVTJebXCflSKIrtflZWoMpHKR5HLoszS8p9iEiWr2K9k0Bk/+PHKr6Is7cbRVynCKJalMPtvRejPZX2CvsVcxrENnko/yWOwFBZZIqqlFJOL63G3dyaWMg662aqqT1kCbAr5IItNm7HHqFp2QDgKmLKo/HtQdUS+qZZ4jf17T6ZVsXHyjWiopZB5VlTC/NlfLGIprm7ubHH94eJHr//2/bkAWxA5zVdVKfyy3ngnXovxp5vbX34EBsvpfMqqZZQugMsPShHIB+xfXRLQXhGO+HcBVqG8qPxaRomn6DrVurL3a+UqSXxw+AWCOoZhdKKEmVNfcXTvrKoI6qJ9W5Sb0hZVlMgOKy33qyVANLC4wWt9HvaE0OA/zW3ApYHP0uRBp/Ov8e1YDBna9DwylOdZDuyRxQ/StJzcL6C1zvjzzQhgBF0vdTqBDIUXZ35gplCTzRxYg47AX5nLOeDbjDu06hGzilCczVk3zeN8Onl6NMmCFY7wYUJj0od1zoQc4kEWjlwDvQI0E4WpkPCNVCTg9gZIFbeGTPMiW/T6hi1YsNfC2JvToLdilcJNDEu0/74RiSwWUIstbn+5+zSa2OJq+oGRjG+nk3Nx43wEldvryfTu05iDrvQ3QDj9PLXFL3dTq/MR0LaYvOFX2NDfwN0uJlOygYPNG2dMHwRw4zDIjcNANw6BdaYXPwLU0B5jdD6Bsi0usdZ/awvzpCeiUCGUcSnFP3o9q3M1/vBhAgiXJPfkGg6xgOhGloUGRNmvthdXRbDSgMssDjx6b+wcb8wUGe/XM0Xp4eyYClYOCzPtP1IWXuiXlQlf3mjnyYAhzR2/WJRwAbOLra9RGgyNkmPagNk3ANm42ewcgFlJLw6Cx8ROmsoF3mOZmhtLdHmfsUIvCnYoegIpRAHiRRFteAtop37Kq2FakGoZjzNfJUQEaF8zqnMR5oXiVW8RoNoGckWVXDmtTCVPEqX8dF+ahLlLCCwLXGvKfeThbwXtudFMfEfbeCCEJ1pf96soDkytqPHoouHUgX/adOjRRff0+uLjsUuf0ykHDomTH+mbMV1ND4jmFWAbmK4+jC8+aYduoeKTfpGU7LwqNK6manlyYu/z/FCMHZhu/rV3QqEzdkJyG71rKviRgvJD2Tszbxy4uVqfVzFjR5yRBfVDqsiyi9OT25sp6Cwmctv1QLihsV3vvIp0oh7T8PD4YMz4HlnXuHbK3LRki3lJy4TMQc5OylrfNcict8vDIkvgzmegTV8O0luc+mYIhbLp4U6J9FOTN+lOkCXMzscXRbbKmenQqAqkDg8MGgPi4U+5UcxmX3Hqj7GT+A9eGOP6koE5svZyLOqTisJTSUbuYqa8t/b2+igfo9gjjGwtZ5VG/1kh81ltoSsJDBqGgqm0tHht1SBDz90KOZ4pXk0dLy7MyYmTr8hnRu4fwPN3CDTbO41LQVNaM4uksABSyVntWc5IltEi1ZhSiWom8AJeMxt+Z2vJ1TG5prwUGkUot/7OYDl9khMZV6Vv8q/DvvYL5VfhqkRtAhsZny8+fIB1YpRQ5uQNuDOmpw/9fr00dn7t973pKW38ysuTN4jjehmrVxdYdGFqPEBc47pfHkPhcQ/Q0qP49lt4ya32ELfpMMfuvKtzHiVCubbED0Nx0vBiEsgFqoQiYdjGdUAl18iwDdiLI2BEJ5H+ro1PVQD+xjR+S+ETSnnNWg214NVF9/puMhZXlF5toYwIROwRgS1OuU5NECc6z1DWQuLxSDpzZIuWoZlEnY/my7JpKvXdMJR+2FuIv2rL0OfeKvyFN8iKN9Y8S4ylu1dY+ZSlcrfnzg8CquCIOdq2YUR1/nBwdgjPNCEFmwYUBnhSbSO02BNxKE2UJ7J87cg7ohgadwr8iCwtzZD0WS3uHorMSNwfsl7tOS0m1u7pYEZxbfg9o01/nqVVlK5k5yWG1k/4WT9hZ33EDRWDZECOJ4az6txOZk0HQq266YyvcdUQpMQ/H901Cz8wwos2HddXaKn7FLO1aXMrguciXWhLoALv/gBWC9Rw7iWAPSIMqgccgSjnGTs05XcxHt+KDK0JdzCM75Vqfiz0VQH5a5dWFc6SS3wiRI0Qwmq2t0bdFS048xOeg/IXfMmMHEYE09D3rGk0iviFdVTaPGuyiS5esniVpB5f0aa7oJsTS9Dv7r9b8AD7OFXmcRNBlxnfam/OjvU+02XZwtkcwC0Htd0mlybSuXXIPjnDhXm9W1aF1Th0LvhGBC197YQ51qKAUvg2H/CVlaW/yyIzCdVQ5BYqAOYnJ35y7QT0R5cklV2qUCbd4EYANmopmUS4imPTvFR7LImF7K1uRat1Ld4T7sJPcWVdHt2DdRmYpWhpZEre5YJfN29wBcdy5sssmkuwgJKWDJzHaIiH1z4KGWtmtVDSuS8Hig3u2qQ52YL8xC3QGSJfJ4qXqAyjFBnXLK0n8BDevSdogB4qcDfBhbRxixkeDmfgnI6f5zINTFP5pVaiTWisVp8F2DrIuJWNfoesstjHWeUv6ElL8HlffXBxhEftghkL79n0n7FBZUGVQO9pQ9aIS8b5pUdIiZqDlkGuTUrFmkc/JWfIA2fkV/51ASDzM3pdLJv+OiqH/aYPqiPZY9mOzC+2cnkJMFmQnZVRDjahOuGzO4B24ZJd/TgTf/XvG/GDYBkwkKkqTiVILHevDk0KeKoNEkRzlSOHmAeQEocNfWrK0FsZ6PWyChrLeb+n16HfOXp6ZKp620YvYtmdP2Uzf//S2fc4KwLvofTumhwFRNR7jNKjZRIZW0kQDPvOyTu0QAd2g5e4oOMeGXBIH4q6wziZEi32h+Zhj3IFrKwgtOtOj32CFGw5paw87UGkYO1DU9egIQ+GPYaunR+XGKWZ5tTRglGh5byzUM5iTamAl7gkVJOzeHNkW+MFEelGfIbK90B50qaCpe4RmcesQIe0pyKMNKtEgIsVAqyicknttBq5UZRoEblYVxVMS1i8hOAinUt1ayGfKRVAv0GWOAh7fxVXHtbNPvW0y3boeM+nT99GYh1ydsQlIReo6cyeSjKfIcyJ1bzbfPG3obg/yr3LOhxgxoksIlnCE/2ZhXRcFK3Fe3L7RGIgSNOEXKLg8dNacKpYQ1WybnWi2g3ElrMdpqE7dRELc/vKFq+cLxkulfDVdr5DTbKrS9FX6vpGkrCVvHwSRLd0/+xao08qDGh0s73cGQcOpg41Xp7qolAeSmLQQSMD+U45QeGyhEnMmmvMRKC+J61msdynZj9g61H6C/2ogF8UMo54lEI6ZQzkom9VQ20kGTuP4rXLk1mabqZZVDZOfC+cM30AE8GqibWhTp126UQUR9VGmBOtdiqjqExqDXEJmGghPh6lxBD3kfJ/IMm8SDIiKHwEJ6xCCAfOabgTKNG2JGOtRH0V0XANMDbLry8lNXOte+CqR3mWBqgOfdSDiGZRiH09cGmIRP6B+UTtGyN4BiziU+24PdzSu3bHA2BVndZcvhA8PZpr4v7kLqaFBT2HZs6gvEstCJc1xiUeLw8zDO2+uqblRgOu5NP3OF2gsl6i1VqlS1giEHeY31kIdp7f8aSuhlGnszTe1Nvu4GSmZoD68DMgJ4OZ9WJC41GgDqZSomYKMPTec4AJYrPEUbXxX6jZm2mC0hwO/p/yt+UiROTFmmV/7MvdcyXGOeYUPCKYD1rBSHdYxrfP3cyy6vTQKGwOpSrRdzU5Srh8u/PIczjVGeCMMwBvZMhpaTU0mAeasFLsePtAG6oDSsAzSwmJ0BrSt1Kch7oATB8k454dbcGojk1qWChNqcYmxC0h7jd8Fs0xXIiuCJ0JDc6EDwPnJNQTBIj5oAZlQT180inJxGAbg0ScoThq/eyBXPdIP5R4FSbSJv8QEqAgK00CsjkXQGhMDHTADKl1sFGOzTOWzlhVYfed8Sydw08ubSrI+VoKHohaJNhvNAF9HqvKAoyy/vWIkKHsF+62kU5Q/CHFDJxeuCtnONvpwDM9j6zredxUex7FpOfp1nprKHSIZfWwc/EbD82NH9z+rG7t6iWMdDCYVglYn5uZVud/Zqv+yfUbAAA="), ("HARDNEG-RFS", "H4sIAAAAAAAC/6VYbW/bOBL+7l/B0wEbqeuwttP2dp26gBsr6WLb1I2dux68hkBLlK1GL15RThO47m+/Z0jJlvPS7fYMQxLJmeFwXh4OaVnWm/7F4Nw9O7w4HbGfmXt++MZFD8vlMssLZl+4/cE7V6E9j1Qhcxmwdudpu82+dtrdVovNZJjlkon0lqWrZCZzp8lG/VP3sPWMXYs4CkQRZWmT/S7m81iyk+Els0/f9s+89i9tp9tg+C1vi0WWsljMPJkW+S1f3rK6VqUq9LNBxCBSQNXx0Yh9ymasELNYKnTk4eKlyJNXJCjM4uClesWykJWztViUspej4aun2ap4qkQoW8+cxqV3w3rspH94ejly7dELiPl3u+2Nj/BR5Cd9B+9qtMihjXfjMKEYvsaX51hlmz2F0YYX78/wzSu9u2zkvvUWLLuWOVuLVpMtnmwYFjlnvoxj1WTqFtZMWL6KJW9UVjdsssYmDVuxkKz4nDE3nceRWlRCqqaWws6zYhGlc9hLBIoF8hqqnbwm/eCIlfYDZ//JowLWkun8xjOG5cVNASJPrZJEwPqfFOgsy2pEiba7ecXRjK+KKG4yGtf6N1kRJbIR5lnClqJYgKQkZkM0K36EBRwKm6XLJujSAJ/4L4NG44174cL8RG17XhjF0vMcnkuVxdfSdvhS5IiIhvtxOAAZUVddjUYgQ+bFmQjsVCSyqTUoA0otpQ/6fcU59XqkrJkoznxtkjq75k7usyZZAAsbZhJj08M51hNx0kHmXN5AvCG0EyMpl8UqT1kCbYcQarS1ZLrMs3mrbcF/tLCnzNoGkEWtfJUiBSyH7f/+yQJYP0wKREXqkcvJF8dsyN9BeC0gbSVuIWf8cdxk7y/x+N39r9N4h+mQmi92fXCiuG2y0/4I7fPXTZNJ5A8O4iF36UEMQ65ZhlwzDblmG3LDOOTEOuSauTHun6Fx+hFSLIoxLNJCXlqN/sW7ETonlmhRnzxSHXovWp3n5m1e7VbZuirfQVuTm2fHsJonRqbaLD12pcHl6WwVxYEXxmLe/tcRoQhN+ozZWQ4HOY3TPhlqXSGe1d1q8z1aTJvkJ52mNc5HFKtWON0gt2ER95ye78dv6FNbWHfTy7z1EDXLyE5kPpf2ybiM6JMx1z1ecaRsE1zzrCB7Cgb8ZYLATZs4CpkNb8H/obWGIzZrsakAsU3Z8+dKFpaDcAWcQ9ZUCwNXCnnkUTMh/YRSEvmrJ+pp6Qg/Cwir2BqdG/aPXn1bWBPFxtLspJMinXKRYh3tJnvh7ASPofgy4IRTXqnRVme/aLXzjlZ3rTY7hbfM2+VChZ1I+n34hti7pnhYdm3VsUztDw4tnL7GAP00YCKObfvD5GrKhSpul9JWRe5wQldAKkjHDw85nBgdrfoVqU4JSbaEQoy0YVDH2tOCSH0ita1Ch1Ua6ue15ewvWttzslugt/Y3yIse+3Cvs9Rm5wZeZPcNhQzePGIi5L0dWjoQoTIRdk0kWHt4h54yis0GY5cqFygXenrP4PQo49gd9GvYGIijOi4O+odHSNu7yHhMXBwAB8539C6TZCfIL0Bbk3Ty1u2flwC5J0pz6pju7ZJOd4462BVkjt2OxlyuCuFftTo2POdyMpBXjpYLGRgqXVjYMI7p9bOY2NcCmFFziHZqra3dW29fw4nbUCf9NtvEEk3m6+Qi2RzbeaLsWlRsQ8dX+6EymPgUGPTi2ALjVNghrFXY6ZKnIk2kSG09WEatUwJNnq2Weg2hZUogKAgIhA6PamOUza7A9dXlibjWqIywsQfOdh3zitPMcH8lg8mcFDbq7a1PsynEFknUzuCrNEKo2s6d/ChoQytpKEWVczcN9EzYxf1JgR1Vzwi49eLcHnX4ckUhMZh8hZyfsKDpNiYmUVrYypk6tAoHJIWcVoHDB1JF87SUlEogZOAFus+uhVWzXPkuu6w/UugYrhQSTAelXRaf7ATCAKNGFopSrVjQZEcaghO4y9rmE0dwe8RuD5psb761ddpW8J0udF2+LXUnFhW7tMc9eQI3X5QuntQ9fjceNztPbjt3tic1RBBQVUlahNalFtLEVKQC9ZhZSqNRFaZ0gUDDU9Kp4tmfohy6eGioUe0uuhihedd+lw4NySwQOt56aM8pEf3NlywLt6Do7NJGa7Kp+cR1L1BDZGGTUYrQkYKCSTndP1KLziFce3EGSirLsbfqTELZK/PI97I0vu2N85V0JlrylIM6DeyOwzUAY59A0W6XyUZFuC5SaD3wAL2omJC5p2t+dP3gfEcOzRZEflGlJzZ9qmL0zn/Pf3rlacZEC/Vkkl1Jp8tUkS0JfA3O3wlaKfwFyWDXSjNdYuO8lvExe8UI9mlkJguUClWgIihppZv9KKoXNDoDoKWgSoNUne70w1liQYCLMPOT5X6I4QPEu219Cc8vcTCiuiuI7fF9WirLzpzmN8fPz5w964QWI1h/+WJTnerYeshRn9tLZ8O+sIWMg0OcNdl6sfkj3UKNpENgRTjXlOhKd30pOuvKy3wiCJS06+a9IQ5omAHLRxD2FvowUPXOHd1MqzZE7cJqYl0rWgvJgtDS7AhktEsoNZC2Q9jxHWg9ZnOz0C2Hv8ubMaexLTE5zucyDdTnCMc7i80tZ4cYoUhgZe1l7O8PbGLiTnFLtGEt1Y/ZSpPcB4lQYLBSs9QXrJiR8H9b/TMZKwm33vGpjuU1iFHblGDrL7LIrw7lodAH8vVB6fODmuyDUvaBkX3w4Kn9YM+5kX+ldllQWSc3lR8BFJ0/qNZY5cFq28AxP4juVoID2ALXB1r3L+vcVIGP7Z3a8fv8pAujGtY2bqekQzD8VDk2UlFq6yU40x0EEaXlTFZiamDIqV5RcJOAATGjFzlROu7wOel0p3sTJ9+zQ/+wWXbbO0XcfQvVxtakXjnyTaCphDyANePO/7n92A+J3zwOZp0HFdoHtMdp6qD2QJLsmzLOYIsIM6PCmWVZAafhMgabjI3ab5WkqrfWkvXSuqixZehZe0YvB3wcpjxr4zS3NGVXs0xYdMxwm4X269756/0D2pJOaEHpEroGsclFuL1Y5D1+tE9b4bTJM+1fpcHYXM8d0k71l+CN5slvvz5nk3WcdX/mnXADxy4i8zllh6/ql3lAhmF/NNKwUGpqwOC0/9vbvdyv6/cD+8IW1mHY7fagF9jTT5jy/l7hR78+701MQUDePMKYaS0ialF5q5Xumdf37S5afSyy+7Cfhry8r7LL2P0bPpLGR99lHPJE/Ub0h12BcFaFpv28wNWG89ch8rf88m2T3vPAPTeaue6c2Ou3tygJP9M1r1fIm8LW97kBLmGVTURNgEyAK9QeThM4o4tVXPToroIU8bMAtWjPWhXh4S/Wg/Ps7oz3Z8HGafFPGRBfH8jp3py6HpNqrK1FVvf7JKyAVyfr2hUB3I9rgy5vhRs1BW+jAWd6HoGO52mo8rxERKnnlVi1tow4wIz52EyQnVzk8+tJe0qhQPc5VZeD6rRtoqLim9pO4395n1q0FxkAAA=="), ("IBLORA-01", "H4sIAAAAAAAC/51ZbXPbNhL+rl+B8j6YbGVaUty71qkyo1hSL1Pbydhu6hmGw6EoUGbNtwMg2YpO99vvWQCUqNi5tuexRAoEdoHF7rMPlo7jvHt78f56dNzrs1mcsAFzryej8eWEOedVuXC6TPBFJhUXfM4GJ/3+a5YVdc4LXqpYZVXJykpx1qdHbMbTSnAWl2tWLosZF16X3Yymk+PeaZf9Ei8WOWfnH35l7vRi9HPU/2HgnXUY/uq1uoekPJ5FECvWfr1m+2kJXldCdSblIs/kPUuXkgtWpWzal0fMHbJpT56x84vJ6Op4ckUDMImiy/KK5p0l7JwNWZ9BfqDE217C/s3eFfGCv83KOe5xF7InllTlPKP1yK6RdfvrFebd76SZwqSiJOdxSULUPWdSiSxRLFG9/qLPVDzLOYb9dp/JmovjNMdKYa3L0Ue2qrKES5bzVLFqqTz2mKl7dn57fRFlM1pZHifoOluzbHZRXY+YS/L3z5Os5h2rdWcRyWIh4jVWnVS4T7E56FpUq6xc6J5plc+PJFudHisRZyUrMHdM8DPpuYquJ1Mm4nJeFfsedZwJSUZthnt+5/37KatWkE1t3+tWaeZfL8WCd1nC81wyGIc/KbZgJwzqIRdS7S+fXS9zfsYmV9EFO6ZLjz1WQnKayBvsit8jk+4kVMKMPX7DZqJ64KXfuarUPS1L8Bja53wFwedv8TVZxflSe6DPfhOZgpWzWV6JeBAZh/EVZJ7sGuWyKGK41u+SzFmxn24+vDnBlpzIOOW9U99xnA48GwOZueTZzF+qLO8yGtJlFSwo1/hSWcE7qYD56ljdo5sdwD7gZyMD/g8njiUr6y76lXPc4r+edzr/nFxP4JLU242iNMt5FHm+4LLKV9z1/DoWiILO5O7DGN2od9ME9T7pDM56IR4F8ENXdzthzmQwcRBu7SYbes+ab0fnvxz3Bo4Xdm5HP3ehYkSaHGsqxLyjI8Xp3N7dkp6w0+nMecpkvHZlE7MiK5ULc6T5Ut4Pb8WSe68ZBvhxXfNy7pJK6Xl2aJRX8dwt44J3tdmsFARMAg2HFvepNSILG+vkVaI3uj1cjy6eDy2qOVzODCYxLn1hYnTxaQ5c+PwJ4k1HtzCSBFdLgUCxs02BDVqbnaaoKiXJEnrTKunzcpUJuN6CK5cQNBq/uya7nTxolDvJynqpHA8WpuEAJCYYwkzLMSLp7z7TUiUWwOeu8MUir2ZGr8eylNFUgWHS9RjPETVBuBuKpzR6L6u1CnoQ9ExfRDcGTmFGRNK0WpbziRCVcFNnQ3q2ZyxWKk4Q0/cxoLeMyzI+IQQb9Ab/OE55rOSx8Yu+PK6Z+wyEPKfZYtMtMs3u9HJyO+qyj3SxVkSI7QBHVI+SfaehReNkH565GjDCAf3IRQselmjsMpMuBsAHbB8MdvF+NAa+TUcY9HHk6dglBVOkmdEpOpS13mtX76OTAmMBpxerSOv2y3qNrfFjqdY1d9E3RWf1aoA4ORyo0ZtGxn80UqunyQ/ZZrvb8wdI6QJ9VQzJBTmAa2YDV7H2cWheFKFWm2NNZp7E0LbfYqCA8cK9G5AWKRIt2ln1o6Si5KxIjLMa7H96h44Sf2mj1MHkow1EbaMNTRuXstj+wYJ34iRSskKWK93YY0MEJaDBpXX7NDk04OL5gGDXw1Lhe1oF0/rO2MYM3Jqd/2aoLcY2XxGwdQ50a6M0mBPv54X1BKQlNEtFfgeI8BIfVw/ZIwjBuzaFtNs+K08HJuHrbOEQOMfzSEEk5r8bt3k4g5tSJGYyK6WKy4S7K+wkkK9LJiBshLW6bFZVuWdjOHV+2mhjrjw/iigGo2hrTLDytm8cz3hOl61oUwsfma0AAGy/0AOCMwcHsUIJa7XQwjvAMxMQXTJGdwdumq+4+5gEabp9hYhGRDUcS3eh8NSEydKCl1gPc5HU2nyJtfmSBxHZrKezsGY3oEeaR+wi1mZLPgDKI5FC2MRED6VX4oPE9ppeNx/2zwTPI/nIef198/Qj1jklxMGniB+QA6Coa/gOaKjCfT4oo13k6TDTAidX2E2ToByk2Ve9PsJnl1etdU7EsvQpGvQQypeTK18bAct1TesDX1N4OqSZInAeq5iuax2OGX2nmWOi9y161nPrl26AW+1k8M1/LZFUbj5AeersCcqJsbte1UZufdsRMpMqXxalHAZf0WcSfEhRLLNF6ZKEoTyM4Dac6EQVlwvu9rvs714IX16U4PURgoM/mUzvz0VVR/NlnWcUVtLF0o3EEusiZx7vDTX2Cw7C6L4FiSqH6Nll99Xj0CFWjNmByGWYOR86BU4OkaqiquTWzi1kGWtkKTXDREBnEsGK4HDHQbO8PHcNulDD1iY2ZxevE7+IV5Fl5838iNKkzoHfW0AaI+bg8XEuX7MNAVphAWxrSHD8dcK/4/vMvXg3jm5ur9+d33bJuw+io+VL4+A/RUgckCtjZpcMbGzdSm1NJMdk5f+Rb1tLe+HUAI5NSYhtpqe+vI9rvrXT3oyaBq3I9iIkPaLbo/CL7vqJvt89es326Mk2mOjWrjKhzJXHxWwesymicIRPkeKzOqPtJNNSeLrTCCeFIvXYtzpc3ZH+vcLG9o2g1WmhfYqc3adDAFAIjgFqYba6zpKHvao72OxuAheeP2k9j/c4ybqQEZwBG+BocO+704AyRJ7VLrp1Wa+rHe7u1IPqPgXA3eTlHhPbwxpc6EVukLl7DnLDx9MAt6kDAeY2ptvpiG5XunVkWz2T7gGRNK6114GhC6H9YShC6O0pBoIfGSdF7xifFPQ45h5FME2mSR775D/F5EdPBD2wkh5GQ8Z+mllTYib6EUmL9aPV7tFOyjiAWptYwRg7z+DDMlrIJbBflhmgyvW+ICGKqNwOqV19nPA8LPJUCSd8zYQwGsxZ1b/WlxtFGbx3iF4CfiRW6K1EIIQeQNJoixTVIGT2mQ91IqDN/BOdDqQbL4X09BQVhNCn0zRmQBZv/z4Y8xndkXLcBJziNBBpqHuLVdg4vXfYHyWUIbP20nTHs0Y/6DUGTUkCxTXmk/0/k4LpE5pC2ldzfabAgughaJKEHWbih6EN5L7JmeEtrqZNQiQVT2lASN5gSxNegBDoh4asWPpJXr/3Y+ujBoQeM5wWF1SvEMJFv25T4vBIoQiO0HYUnvmn6Za5xVIqlJKoQNDzf/zxDDIKbmshuwoMNdkDCIo4Vi4pfkE0NZP0V+n2MK2IQE86bFTp2ffYvAKkoKxF9RmBY2Kyq8icEUWFz9Y11SRMjcJpE65xl2jB+Xv4o2VbphjRxKDqkZeigODTl6UNf4kEGWqGPrm+6x6wIvkA3BVgs1mJK51xed4MvLD1sGu+QJ6ROE8/H1frFSf0GOtraJei8oWY3yRxzoVJWbAyrZEWY9hka7NNWWgz9unGuDEOIsuSzrE+sjvxVqLxZswNzndIouDZUudo8NvkoTcgr5zocIjsU6tjdH15o0GWqklAS539sXF2h5zmFnU9gk3qdtHqZr3zhV69i5elvTRi29FzMaUC5Y4VVVha8HanlzLnmm+NFQo1oY1nxJAKzt83P6kykWisaNsYdiKxd/sQzvWx99kOuudD1NCIEDxFCGsx/L7X65nBMvFxZi0lFUMhifT662dIbt1Wbwn4PY8LOZSq2yqHDmmjCeKGkEjXCHsIsqRohyRNlx7om/YTAo1hLny6RihG7B+9xDvptMRFwms1NNCDkbsmGk5H88iwsWGD2mNFh0ltAc/sB2BMHp7ATcUIeAKEIs95nhG1+cG+b7jIwGNNRqPjG3Hdsa+v3v+b4bimdi1Y/yKzUFLBXt0MfF1SdXd9v6G+u8AITG4MdSHvMHFIygdmlqF2OmBTvtaIIeiQqPiB8+Fny/l2kshwAdmKZGA7W7mnBfAvBQXFS2vVWhAYeAL6nROND7UFkrAtcvykCZw9jnz77SZ1Vptsi/BbaXWZPfxyFFG5oHxPYq0EIuBm4lSaNCDojkFrNiWok2V9C2zgMDtji8CK3i25UVA+V9DIJTtoskr+qh/o5ks03foLUS3r2drVKOd42mjScoA9CDqfSuw2FdAnk2t96qXuSG36JULaVNO7ppLufSodnJQvfUH1OXegUZJO2+WiqTdQ9Rr6dZxSshuiwsAR782YVy1ktYX54cauh1YcC1N+MnC4A7zWzs2JgFwGBlZD0NrLAINAEZpJsf/59zf2hlFuI0UzrpRNFPT3mGlov202/HF4q0Wzn9hto8/70rD+ozlt7eNU55ZP5QZDcQCR5m3C3rTfUeDrZ0a9h2PcvJVuwBLMGwzzGE9pXs/zUWPtwOQxJzRztbZHPlXxcN42NokZUi2IgnR1WL45VLGr5RhN85yC7UCV2R5ctR5bKTA0g2xbrt15rmvOCc4ggD3205Ad01uUVpQ2b1N0DQBvU5yWCwWO5Sy0IKpKueZ3O31/Ks17QIHyOHP1q5u/9M6GDH/09vr9LyiroDHJK8mPqHpl16ELVkdEr2wDjYTGVzjGiOozGvavdtjnDK9PzAtJ76jZIlMcadVGHF0r2eBlxvbgPQ+5Eb0aMqU7Xeeb472MdKmTwfhSDVHlQCKPl7lC6qOTAC+TCmeDxdBZqvT4hz+jdf/K6VAnrOn4v1dZ6eLNCJXDqOFrGoz9tUBLGxmJQtWSBZsWaURwqt6Z30u3MtTVdxi3KSXqo3AUFahWR5FjwnvjGGkAWHOzDehNUiwWq6Af0tZQOm2aPERy3+xSMy50vc5/AckxkrQqHgAA"), ("FACTOR-01", "H4sIAAAAAAAC/61a73PiyNH+zl8x0dX7WroFLeD1JcHLVnE2vqRib1yA816Ko1RCGoF2haRIwouLI397nu4Z/QC8m6u84QNIMz09PT3dT/f0YBjG3ehm9tdJp9sT5mQ8un0Yi0yuwryQmfRF7/JtryeWMkgyKdz4RcTbzVJm1kAErlckWS6SQBRrKYovCZo2YRRKbnu6EOZdL79oi8nd1GoTTcyEeBVplnySNLwVYI4vYbEWRCtk7CW+zMQySrzPOeS5+9myxXR0N+5034lnNwp9twiTWCRx9GKLv7irVSTFzeMT5rof/eT0/vDOGrQEPulLsQZd5C4dGRfZi52+iHqhmUyTrBDqY4JIgK8r3ojZ5VR8SpaicJcR1vFGZMHuvZttPhCjIIn89/kHWp2e7VKEsXg/ffzwNtkWb3M3kN13VmtayFR0hekWkZu3hS9zLwvTSm7oLi/c2HejJJZiPJ4QQ/kssxe0Z9LdYEiWpB3qDZXOoBwRbHNohnSKffDFk4gwJoJmSX1h3PEwWd7ykiyTESsph+ZIkr4wV24hsRFqd3NonmSBqsSTsxNDcTPq3D1Nx+b0B6z4b72eM7vEQ5HdjCz8lr1Fhq1zdpZwc9rE2dNH7ErvWkzH9+ARS1iM34KuhLdOQk9irSz7SngyipRNOLtrqMXiIR28ul3xYSjedO3LNj28e3slSMlQ2lpGfgdKFU+ZvxVvxZ/C2A+JptO1+6wAc2mJiaM4TWpOV+1Wk5Mwc3cjRRrCntgUlZG+CNY+jOtjQtpbwSRcUPvyGZPd/IivMcxty4q0xf9lYQFzUCbvKOuxi13RLpvy7Wbjwso+5SA3DKMVbtjC1E8ULu1tEUZtQf1tkb9AliLcyFaQJRuRusUaJJpYPOK1HA9vg+FC4XHaBh2MJqe31G+1/jSejKF3ojYdJwgj6TiWnck8iZ6ladmpm8HyW+OfH29BRtRlU6vly0A4UeL6ZgzttFkC7Th5Kj3QHwtuU6tDwqqJ4J6smeZwHr05H7pJ/C2G8GBiY9KXdc0T2SSDzGy5A3tFaG4Up0wW2ywWG0j7AKZKWiML8mIby27PaAte2Vth1LZo0Gu2jeHshiWOPt+R0m0SVJjj/rhdggr8J3dfMG7286wt/vqEr7+M/94GVkzx+PHHFpH+0FaA0CTAqJoKAj7YoHywifbB1tQPNtM/2DziweYxD7Ya9WCD+2z0U1s83v0MBoYyJSwMq9wZre/Q9pkB7u1yG0a+E0Tuqvf7S0Ky0eRheiXMGqfPnPvSahEReMwNt0tMR/f39ONd0fcdKRAzZpHuihJ6SGWketVr1bsO6SEPVKd6C5fPip+i8q52xqJ1MyMsoBlbN+P7+6nzE0sgY0fuCrEiwud36ndLjr3CGGWOG5mtpHkz02Z4M7O5xSkuc/NkL3/b5zvR757ieeeD8IpuL+tXWM54qmA3LzGPBVglBYnuAkYy4RIQsz7DQJjYVBhMYOyxcYe9eyhjQ48c7B9bWRgWLBo7A8kXzAyjYvCjjVfLow/gWsLFeaIhcwecGADQvIR9LfUeJAfxu2EzLu+J/mC0GoxI9xgIYmg3ToSXIPAlkeKiKGktzD1zY2i71xY/WLVAMyw49W1CQkevpFprQ237/FAvtBpcqQnz1yzp8/gNtqcqfJ13Y5GRjM1HixRGTzOLQ4EbRab5OP+8sN28eEmliQ21bMJvaA+ks9e7LJsGWiz6ZxIdbmq1G1JxCBGQaSCy5EtuHAlEozwaZRrFJRl0zO4RPxvW8fpZtfOaq7P3DsaCoPusUQtW74hdJOc6A2gcvqItIIwZGOw5EJwIB4Itim3IOIJWtGjf+5SbqRZa9/mhV5gb6cbDLNnGvhkAfgsztanJtKCkS0484nwYxoVppuKD6Fo2oiB3cuQdzptDn3kMK+2ZlJYuLD25iqamnr/odqEZio02fZlK4vHtqI4B0ncvG/CPvs7l3ejhFP+vaZQNWGdwxq/GlZqRV4C2wenmfjz6+FooUV5GahzWOMWN0z6in8wQ1alvbCOx8z53+yZWO7ZJDY7u1Qu5VVScKJrYHtV61yOg5uxrbFf519wgNDIWjY2FLetN3ZP531oHIFboRrnWOWwVupS+SbuSW7W/37Iw9jYOYSzYo0NbmwV9H65FEwX3EKc0FS+JaGF7dyCaxsoG33hn02++P8PAK0TgOSr8cdvCY5mIt428apObDY+p3MrLj90IGOqSMwPmzv1LqYfkRR7trqSAO0FJPVjT98id7DAPwhhzmbdzb1G5v7Lmgd0LDuJ/hKI4gR2ihwJ4GFKfKHa1QYNp7MbMocnTsqrw8zD6G8MGJ+iUyroxElARSBcuJge0/u0GGQ5YICnORcOQdQiCA6Ws/sBQyTd0awxIfV9VpNJz8hmj/jm2N+4zpw1AAxhLtQWrcqSaodqE3xpYG8cXDbJIRdLI9QA6wITQl/o8o84seRqFRa3W2/mKVKoUeLTvLJO24FOLPd7zQoKDpiGjyK1T6OSZkF96czr68IzIKZwoM6d9O92SE9/O/wk+/wttLSovnivPWVikIgskhVyUrm7fyjxcxZqTOvA4PreZDSBoa7XqTaRsWq/Jq6371qZTUbU8sm7PlrGfk7GYBhIjSyM6ThZkApwYGArNKd1SwR0N8AdK5Xq5ox0YbeTBrZbarM7/44OdpqMsZqWjrDDPjrIVNhm/xNiG6Wz8iJOvpnbPT7opmQOLKcwkCdrK+nE0ZPi6FhedC4J/5D6aCokKCgEZmVNOaW6+Tr7EdAQi1+4QT6sMfZSvEZK+IajS/gLlHBrQRK/H4MUtR/ClWp417pLPk/o9FDy+6ff11uaFcsIZRzJKvszbNnNApWGz9F32vyHeV01UwQ9eq9zkiJ9a4YgYzro2m9cSOicTMjSM4aQos9BzaFuGs2wrrXle6NihjJYkMDoGmdqoSzMr+EPovuoKGeVSGG+M82XwrKOu7aZp9FKuAJAz0FuAR2JJk8zxbNORcEE+SXMxW7TWhgJOKi/oW/YMSRl7D30NkSNOsQSZ86tlUe4DK8Dh3Gz4Ao4zZF1ObVicS1VsL3kc5y9W65vmuYR5UqFFkHFqc+Pz+EnFRdc/dIUD5sXo4QMhiIirMM2KjDC5nkUlidI0aRaN4xjS2e8IxecNNCCzJQSgJH+nIudOtx/KBMhGNuKQ95M1HcHNnrgqrycznmizH5z6AVDu++9ZlkOVVtmu71P1gpgGxlPlMXPm2WSnYdAHMS2lDuilpDVCn/PlNbdJoFIBr7Dmuh0wYR1QlgQe3iZV4+tBLBZyu3QLuu0p3VPFulxJzblMEcpovN8N3vf6B97EldgTn2CDHDewDr/EQuwvLgb9bn7Atoqj/i2Srl/r+tR+vT1QEfG96B4jFygQAxoCkObmOzJWtk7MO6TEO4AhPfHTFk9ryhmfhuvtucXTzjmlvROzb5u3B/NulgbFNJUwH4Lb48KcqTAYCKuTyJ3uwC8NlNUqJhut8C9UZiSVv4q3sLOVV3v8ZFN7fNOlr49WxyydFS+uGnHuzP+NeNYvoyiKhxAVtdHXFNln//dVnlyBwA3Ew7H5BAtwtuHD+abW1bc9lm3fPfZR93CaNKuc93ec8x7q9K3q/5bHMb8TP3YrV0MRlAsz1LlQO6hGHE+guyavdbXKksG/CXGIq97hVwT76njbiG2Q49BIIihFeCUvsAa/xAZV539z6APfRTPOnIWS2Ypi6Xym0jD0nSRfi3oqEsHQLFXIbLimlJnD3sKG+x8J+HrECpQpStdbk77FMw5mZAZn3vuBgYdIlrIo+GqmjES/ikkZk84q39pMKSE7QnPe4GbZ69QQF7XVfQWEtek1QBh4nYE0OyWdVKSTr+I1zp7vfyAY/iYEl5ANykmDMjulzJpTYPFztwLkV0AYZVVuzMrGyXCdNbY/MJ7zEvSoyiWzM0we3w9KtGjcjZiQRuOtyAciSr6ARBn9qkMbW94LPTl8/UT+oyCa9h5XFiVYl/u4dcmfv+LF1+r+43ijo+dIVa8gIgVW/EyMk4N4pijIeUHBGqCybfWypluZ03rXLbk8uB9+3WdKL42j3m8/5tEJb7biExx+MJHckQUCPbdu6YhqYSCUdujvNiGa5v1Bud7yLEc7g/eK8+bfnR3/86WzuBsyaKizWn+jGftDsmjdXJ+MmLw2ZHI05Mw/VI1SLRU1D1aIGiBQBophKCs2pr20WUEDux+g4qPAZC/nwYV204sFd1XIgPrC2j32WG0rx569BN3ynG7yil9HCWhDqnX17WWSFNgF3EjhwGCqYkg+3Fe8OWTgxJLJwNH2WTZ5SLcd42C1617VRIkHXpZwEDz/OPz4oz6El+vI6IZJrYyupLOhfakpaAUQKTLTZXUwGgr7St01pstGhZOvKpv1VnF0qcnSDxog5J6CkEvvN3/+45XOCGjiAjHITcV8HyWDN7QPiKfrUD0u6OJif/E4mk4vCIqxHD5YXdyN/nx/cajr0hDF1Beq3a/eqOpL1Op0Uq3h6EL1bBXL01UsD6diLY/EKkW5OhGlnBf1NjdiJ5/XHn3kx9oOc0rDhrRq2ovV8kjqJQEnIWuFjnoayM4zHHSCq70As7HFq76LRVUEVCpXXlESKbdokpwq7afRbIzdZ8noQKnUQYrhVPNSmGqZE77ju0ASo97fCPwzor5iVleorEW93lqTfGsVJTlftFK8pjTionQplYaQX1aBjMFvqK+8ST/OFWlkyDOj8DtkKoprro5rQ3IHL/zjVVm0J0elEr96W4f0RkUylm24wrjWV6qDy5r7suK+rIdSo35WP22lEUqUlYSnSvhI6YoyBeTQ6/4r+YY+tKrD4bFRU6EMd49W6ZgwblA1Dbt/ath90i1zOblzad7yI7f7Qv8KcIi7yff+Pi7rUbwAUVtQvIqLYY/+/hG426gY0sVTW/3DBXno0NgWQecPWtqTeeo/GBzPgpTCsD8lcA++0KA/Z1DT17gqHTDL8h8vxKzAH3Hm+8YVC1SCa5eB3Q0O+YKW3sIOOA7hsuNwJcdxNm4YO46hwt3eUOyAxerhMKfbdTdbPc97C9o/up0om6jC1NMFJj1uYVqtfwF+vIU0eiQAAA==")):
    p = LAB / "Experiment" / d_ / "run.py"; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(gzip.decompress(base64.b64decode(blob)))
r = subprocess.Popen([sys.executable, "-u", str(LAB / "lab_entry.py"), "FACTOR-01", "report"], cwd=str(WORK), env=ENVV,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
for line in r.stdout:
    if "Warning" not in line: print(line, end="", flush=True)
assert r.wait() == 0, "RFSTUNE-01 report failed"
for n in ("factor_report.txt", "factor_summary.json"):
    shutil.copy(SP / "out/safe04" / n, OUTD / n)
print("FACTOR-01 outputs:", sorted(p.name for p in OUTD.glob("factor_*")), flush=True)

## 4. Summary

In [ ]:
print("[cell] summary", flush=True)

prog.update(gpu=GPU, torch=torch.__version__, lab_data=MAN["created"], e2e=E2E_MAN["created"], finished=time.strftime("%Y-%m-%d %H:%M"))
PROG.write_text(json.dumps(prog, indent=1))
print(json.dumps({k: v for k, v in prog.items() if k != "code_md5"}, indent=1))
if (OUTD / f"{TAG}_report.txt").exists():
    print("\n".join(l for l in (OUTD / f"{TAG}_report.txt").read_text(encoding="utf-8").splitlines() if "DECISION" in l))
print("NEXT: download /kaggle/working/e2e_out (report, fold parquets, final models, fuser, zip). Submit submission.zip to CodaBench "
      "ONLY if the DECISION line says PASSES, and read it with the rule registered in the experiment README.")